# LSM-ECG thesis — Task 4: a longer window (the neighbouring beats) and the ladder
Each beat is classified from a 1.6 s window, so the timing of the neighbouring beats (what tells SVEB apart) is often missing. This notebook cuts each beat at R ± 1.5 s as well, from the same filtered records, so the previous and next beat are almost always inside, and runs the full ladder at both sizes on the same beats: **no liquid** (the encoder's spikes counted in windows), **random features** (those counts × the liquid's input weights, ReLU), **the liquid without its internal connections**, and **the full liquid**. Two questions: does the longer window help (especially SVEB), with no hand-made RR numbers; and does a layer (random or liquid) gain more from it than no layer does, which would mean it combines the timing of the beats in a useful way. Windows of 20 steps cover the whole input (the paper's rule: 577 samples → 32 windows; 1081 → 58).

**Shared by all four task notebooks.** Config 11 (morphological filter, robust normalisation, Multi-Threshold coding, linear SVM readout) and the paper's liquid **with the three fixes**: Gamma weights drawn as described, A_in measured on each split's training rows, and the reset of Eqs. 17–19 (lsm2's simulator on the authors' weights). Splits: DS2 (train on DS1) and the authors' 12 CV splits on DS1, the authors' readout seeds, the replication's liquid seeds. Before its runs, the notebook recomputes three of the replication's stored runs (the liquid with the authors' reset, random features, no liquid; first seed, first CV split) and checks the predictions match beat by beat (`VERIFY`; needs `REPLICATION_DIR`, skipped otherwise). Results: `WORK_DIR/<task>/summary.md`, with csv tables (and a figure for tasks 2 and 3). A restarted session continues where it stopped.

**Run time:** about 7–9 h with 2 fit threads (needs about 16 GB of RAM); about 6 h with 3 threads if the machine has 24 GB or more.

## 0. Constants (edit here)

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  CONSTANTS — the only cell you normally need to edit                 ║
# ╚══════════════════════════════════════════════════════════════════════╝
# Task 4: a longer window (the neighbouring beats) and the ladder Results go to WORK_DIR/t4_window/.

# ---- Machine -----------------------------------------------------------
MACHINE = "local"                  # "kaggle" or "local" (your own Linux machine; set its paths below)

MACHINES = {
    "kaggle": {
        "INPUT_DIR":   "/kaggle/input/datasets/tienthinh22/mitdb-1-0-0/mit-bih-arrhythmia-database-1.0.0",
        "WORK_DIR":    "/kaggle/working",
        "SCRATCH_DIR": "/kaggle/working/scratch",
        "RESUME_DIRS": ["/kaggle/input/datasets/tienthinh22/mitdb-common"],
        "SESSION_BUDGET_HOURS": 11.3,
        "DISK_LIMIT_GB": 19.0,
    },
    "local": {
        "INPUT_DIR":   "~/lsm_ecg/mitdb-1.0.0/mit-bih-arrhythmia-database-1.0.0",
        "WORK_DIR":    "~/lsm_ecg/output_t4_window",  # a new folder; results go to WORK_DIR/t4_window/
        "SCRATCH_DIR": None,
        "RESUME_DIRS": [],
        "SESSION_BUDGET_HOURS": 240.0,
        "DISK_LIMIT_GB": 250.0,
    },
}
_M = MACHINES[MACHINE]
INPUT_DIR            = _M["INPUT_DIR"]
WORK_DIR             = _M["WORK_DIR"]
SCRATCH_DIR          = _M["SCRATCH_DIR"] or WORK_DIR.rstrip("/") + "/scratch"
RESUME_DIRS          = _M["RESUME_DIRS"]
AUTO_RESUME_ROOT     = "/kaggle/input"
SESSION_BUDGET_HOURS = _M["SESSION_BUDGET_HOURS"]
DISK_LIMIT_GB        = _M["DISK_LIMIT_GB"]

# ---- This task ---------------------------------------------------------
TASK_SEEDS           = (2026, 69, 420)   # the replication's seeds (liquid weights)
FIT_THREADS          = 2           # readout fits side by side, one core each; each needs about 4 GB of RAM
PRESCALE             = True        # scale each training matrix once (faster, more RAM); False = batch by batch, same results
REPLICATION_DIR      = "~/lsm_ecg/output"   # optional: the replication's output folder, only read, for the check below
VERIFY               = True        # first recompute three stored replication runs and check they match beat by beat
                                   # (skipped, with a note, when REPLICATION_DIR is not on this machine)
# Windows of R +- 0.8 s (the paper's) and R +- 1.5 s (the previous and next beat for heart rates above 40 bpm).
TASK                 = "t4_window"
WINDOW_HALF_S        = (0.8, 1.5)
WINDOW_MODELS        = ("no_liquid", "random_features", "liquid_norec", "liquid")

# ---- Session control ---------------------------------------------------
SELF_TEST            = False
SMOKE_TEST           = False       # True = tiny end-to-end test of the logic (few beats, one CV split); the numbers mean nothing
BENCHMARK            = False
ANALYSIS_RESERVE_MIN = 25
RETRY_FAILED         = True
OLD_ANALYSIS         = False       # True = also rebuild the replication's tables and figures

# ---- Software ----------------------------------------------------------
PIN_TORCH_240  = False             # runs on the installed PyTorch
AUTHORS_REPO   = "https://github.com/Richard97Code/LSM-ECG-Classification.git"
AUTHORS_COMMIT = "02e45c4f3232886aed16a40d77de1f851ef4fcdd"

# ---- Experiment scope --------------------------------------------------
SEEDS = list(TASK_SEEDS)              # read by the reused cells

OLD_GROUPS = ["checks", "brian2", "main_final", "ablation_final", "rr_final", "baseline_final", "gain_final",
              "main_cv", "gain_cv", "ablation_cv", "rr_cv", "baseline_cv", "nn_tune", "retune", "kstale_final",
              "kstale_cv", "huang_final", "huang_cv", "meanrr_final", "meanrr_cv", "elmrr_final", "elmrr_cv",
              "rrliqmean_final", "rrliqmean_cv", "meanrr_nn", "hybrid", "working_diag", "working_final", "working_cv"]
GROUP_ORDER = OLD_GROUPS                     # read by the reused cells only; this notebook runs no replication group
GROUPS_ENABLED = {g: False for g in GROUP_ORDER}
SHARE = "BOTH"; SHARE_GROUPS = {"A": [], "B": []}
MERGE_DIRS = []

# replication settings that its cells still read (unchanged values)
GAINS = [1, 2, 3, 5, 8, 13, 21, 34]; DENSE_LAMBDAS = [1.5, 2.5]; RETUNE_TRIALS = 30
RETUNE_GRID = {"gain": [1, 2, 3, 5, 8, 13], "lambda_val": [0.8, 1.5, 2.5], "num_levels": [50, 100, 200, 400]}
NN_EPOCHS = 15; BRIAN2_BEATS = 10; NN_TUNE_TRIALS = 30; NN_PARAM_CAP = 35000; NN_TUNE_MAX_EPOCHS = 30
NN_TUNE_CHECKPOINTS = [5, 10, 15, 20, 25, 30]
HUANG_MATRICES = 15; HUANG_DIM = 50; HUANG_C = 1.0; HUANG_GAMMAS = [1.3, 1 / (2 * 1.3 ** 2)]
HUANG_THRESHOLD = 0.8; HUANG_SVM_CACHE_MB = 500; HYBRID_RULES = ["replace", "add"]
WORKING_LAMBDA = 2.0; WORKING_GAINS = [1, 3, 10, 30, 100, 300, 1000]; WORKING_DIAG_BEATS = 300
WORKING_T_PERTURB = 150; WORKING_MAX_FIRING = 0.2
SAVE_STATES = []

# ---- Performance -------------------------------------------------------
SIM_BATCH_CV    = 4096
SIM_BATCH_FINAL = 2000
NEW_SIM_BATCH   = 2000     # beats per GPU batch for the liquid simulator (results do not depend on it; bigger = faster)
PIPELINE        = True
FIT_WORKERS     = "auto"
CORE_SEARCH     = False
RAM_RESERVE_GB  = 4.0
READOUT_JOBS    = 4


## 1. Environment

In [ ]:
NOTEBOOK_BUILD = "9b0d5a855d73"   # fingerprint of this build's code cells (do not edit)
# Environment: installs, imports, versions, device. Smoke-test overrides.
import os, sys, subprocess, time, json, platform, shutil, glob, hashlib, traceback, math, re, gc, random
import importlib, importlib.util, datetime, itertools, warnings, contextlib, io, heapq, collections
warnings.filterwarnings("ignore")
SESSION_START = time.time()
ON_KAGGLE = os.path.isdir("/kaggle/working") and "KAGGLE_KERNEL_RUN_TYPE" in os.environ
NOTEBOOK_BUILD = globals().get("NOTEBOOK_BUILD", "dev")      # fingerprint of the code cells, set by the build
if SHARE is None:
    SHARE = "BOTH"
if SHARE != "BOTH" and SHARE not in SHARE_GROUPS:
    raise ValueError(f"SHARE must be 'BOTH', 'A' or 'B', not {SHARE!r}")
MERGE_MODE = bool(MERGE_DIRS)
if MERGE_MODE and SHARE != "BOTH":
    print(f"merge mode: SHARE = {SHARE!r} ignored (the merged result is checked against the whole plan)")
    SHARE = "BOTH"
if MACHINE == "kaggle" and not ON_KAGGLE:
    print("NOTE: MACHINE = 'kaggle' but this does not look like a Kaggle session (paths may not exist).")
INPUT_DIR, WORK_DIR, SCRATCH_DIR = (os.path.abspath(os.path.expanduser(p)) for p in (INPUT_DIR, WORK_DIR, SCRATCH_DIR))
if isinstance(RESUME_DIRS, (list, tuple)):
    RESUME_DIRS = [os.path.abspath(os.path.expanduser(p)) for p in RESUME_DIRS]
os.makedirs(os.path.join(WORK_DIR, "logs"), exist_ok=True)
import threading
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")   # deterministic cuBLAS
SESSION_TAG = datetime.datetime.utcnow().strftime("%Y%m%d_%H%M%S")
LOG_PATH = os.path.join(WORK_DIR, "logs", f"session_{SESSION_TAG}.log")

class _Tee:
    """Everything the notebook prints (all cells, all threads, warnings and tracebacks) also goes to LOG_PATH,
    flushed line by line, so `tail -f WORK_DIR/logs/latest.log` shows it live from another terminal."""
    def __init__(self, stream, path):
        self._s, self._f, self._lock = stream, open(path, "a", buffering=1), threading.Lock()
    def write(self, text):
        with self._lock:
            try:
                self._f.write(text); self._f.flush()
            except Exception:
                pass
        return self._s.write(text)
    def flush(self):
        self._s.flush()
    def __getattr__(self, name):
        return getattr(self._s, name)

if getattr(sys.stdout, "_lsm_tee", False):           # notebook re-run in the same kernel: switch to the new log
    for _t in (sys.stdout, sys.stderr):
        _t._f = open(LOG_PATH, "a", buffering=1)
else:
    sys.stdout, sys.stderr = _Tee(sys.stdout, LOG_PATH), _Tee(sys.stderr, LOG_PATH)
    sys.stdout._lsm_tee = sys.stderr._lsm_tee = True
try:                                                    # a crash's traceback goes into the log file too
    def _exc_to_log(shell, etype, value, tb, tb_offset=None):
        with open(LOG_PATH, "a") as f:
            f.write("\n" + "".join(traceback.format_exception(etype, value, tb)))
        shell.showtraceback((etype, value, tb), tb_offset=tb_offset)
    get_ipython().set_custom_exc((Exception,), _exc_to_log)
except NameError:
    pass
_latest = os.path.join(WORK_DIR, "logs", "latest.log")
try:
    if os.path.islink(_latest) or os.path.exists(_latest):
        os.remove(_latest)
    os.symlink(os.path.basename(LOG_PATH), _latest)
except OSError:
    pass
print(f"session {SESSION_TAG}: everything printed is also written to {LOG_PATH} (and {_latest})")

N_CPU = len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else os.cpu_count()
if PIPELINE:
    # one fit per core: the maths libraries must not start their own thread pools inside each fit
    for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
        os.environ[_v] = "1"

import importlib.metadata as _md

def _ver(pkg):
    try:
        return _md.version(pkg)
    except _md.PackageNotFoundError:
        return None

def _pip(*args):
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], capture_output=True, text=True)
    return r.returncode, (r.stdout + r.stderr)[-3000:]

# Kaggle's core packages must not change version mid-session (a NumPy swap after it is loaded breaks imports),
# so every install is constrained to the versions already present; pip then picks compatible package versions.
CORE = ("numpy", "scipy", "pandas", "matplotlib", "torch")
CORE_BEFORE = {p: _ver(p) for p in CORE}
INSTALL_REPORT = []
if not ON_KAGGLE:
    # Your own machine: you install the packages (in a virtual environment); here we only check.
    _need = {"numpy": None, "scipy": None, "pandas": None, "matplotlib": None, "torch": None, "snntorch": "0.9.4",
             "wfdb": None, "tqdm": None, "scikit-learn": "1.7.2", "joblib": None, "threadpoolctl": None}
    _miss = [p for p in _need if _ver(p) is None]
    if _miss:
        raise RuntimeError(f"missing packages {_miss}: pip install them in the virtual environment, activate it, then run the notebook")
    for p, want in _need.items():
        if want and _ver(p) != want:
            print(f"WARNING: {p} {_ver(p)} installed, {want} expected")
    INSTALL_REPORT.append({"package": "(local machine: installed by hand)", "status": "checked", "tail": ""})
    if _ver("brian2") is None:
        print("WARNING: Brian2 is not installed; the brian2 check will be marked FAILED, everything else runs")
if ON_KAGGLE:
    os.makedirs(SCRATCH_DIR, exist_ok=True)
    def _constraints(skip=()):
        path = os.path.join(SCRATCH_DIR, "pip_constraints.txt")
        with open(path, "w") as f:
            f.write("\n".join(f"{p}=={v}" for p, v in CORE_BEFORE.items() if v and p not in skip) + "\n")
        return path
    if PIN_TORCH_240 and sys.version_info[:2] <= (3, 12):
        code, out = _pip("torch==2.4.0", "-c", _constraints(skip=("torch",)))
        INSTALL_REPORT.append({"package": "torch==2.4.0", "status": "ok" if code == 0 else "FAILED", "tail": out})
    elif PIN_TORCH_240:
        INSTALL_REPORT.append({"package": "torch==2.4.0", "status": "skipped",
                               "tail": f"Python {platform.python_version()} has no torch 2.4.0; using Kaggle's torch {CORE_BEFORE['torch']}"})
    for pkg, need in [("snntorch==0.9.4", "required"), ("wfdb", "required"), ("tqdm", "required"),
                      ("scikit-learn==1.7.2", "preferred"), ("brian2", "optional")]:
        code, out = _pip(pkg, "-c", _constraints())
        INSTALL_REPORT.append({"package": pkg, "need": need, "status": "ok" if code == 0 else "FAILED", "tail": out})
    for r in INSTALL_REPORT:
        print(f"install {r['package']:<22} -> {r['status']}" + (f"  ({r['tail']})" if r["status"] == "skipped" else ""))
    bad = [r for r in INSTALL_REPORT if r["status"] == "FAILED" and r.get("need") == "required"]
    if bad:
        raise RuntimeError("Required install failed (is Internet ON in the notebook settings?):\n" + bad[0]["tail"])
    for r in INSTALL_REPORT:
        if r["status"] == "FAILED" and r.get("need") == "preferred":
            print(f"WARNING: {r['package']} could not be installed; using Kaggle's scikit-learn {_ver('scikit-learn')}")
        if r["status"] == "FAILED" and r.get("need") == "optional":
            print("WARNING: Brian2 could not be installed; the brian2 check will be marked FAILED, everything else runs")
    changed = {p: (CORE_BEFORE[p], _ver(p)) for p in CORE if _ver(p) != CORE_BEFORE[p]
               and not (p == "torch" and any(r["package"] == "torch==2.4.0" and r["status"] == "ok" for r in INSTALL_REPORT))}
    if changed:
        raise RuntimeError(f"pip changed core packages {changed}. Restart the session and run again.")

import numpy as np, pandas as pd, torch, snntorch, sklearn, scipy, wfdb, joblib
import torch.nn as nn
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import SGDClassifier
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import confusion_matrix
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedGroupKFold
from scipy import stats as sstats
import concurrent.futures
from concurrent.futures import ThreadPoolExecutor

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TORCH_PINNED = torch.__version__.split("+")[0] == "2.4.0"

def gpu_supported(cap, arch_list):
    """Can this PyTorch build run on a GPU of compute capability cap=(major, minor)? A GPU runs code built for its own
    major version and a lower or equal minor (sm_61 runs sm_60), or PTX (compute_XY) at or below its capability."""
    for a in arch_list:
        kind, _, num = a.partition("_")
        if not num.isdigit():
            continue
        maj, mnr = int(num[:-1]), int(num[-1])
        if kind == "sm" and maj == cap[0] and mnr <= cap[1]:
            return True
        if kind == "compute" and (maj, mnr) <= tuple(cap):
            return True
    return False

if DEVICE.type == "cuda":
    _cap = torch.cuda.get_device_capability(0)
    _gpu = torch.cuda.get_device_name(0)
    _archs = torch.cuda.get_arch_list()
    _hint = (f"Install a PyTorch build that includes it: on Python <= 3.12 'pip install torch==2.4.0' (the authors' version); "
             f"otherwise a CUDA 12.6 build, 'pip install \"torch<2.15\" --index-url https://download.pytorch.org/whl/cu126'. "
             f"Older GPUs (e.g. Pascal sm_6x) are not in the CUDA 12.8+ / 13.x builds.")
    if not gpu_supported(_cap, _archs):
        raise RuntimeError(f"PyTorch {torch.__version__} (CUDA {torch.version.cuda}) has no code for this GPU: {_gpu}, "
                           f"compute capability {_cap[0]}.{_cap[1]}; this build supports {_archs}. {_hint}")
    try:
        torch.ones(8, device=DEVICE).add_(1).sum().item()          # a real kernel launch, not just the device query
    except Exception as ex:
        raise RuntimeError(f"PyTorch cannot run on {_gpu} (capability {_cap[0]}.{_cap[1]}): {ex}. {_hint}")
    print(f"GPU: {_gpu}, compute capability {_cap[0]}.{_cap[1]}: supported by PyTorch {torch.__version__} (CUDA {torch.version.cuda})")
# Plain float32 everywhere: Ampere+ GPUs (e.g. RTX 3090) would otherwise run convolutions in TF32 (a T4 cannot)
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.use_deterministic_algorithms(True, warn_only=True)   # PyTorch's deterministic algorithms on
if PIPELINE:
    from threadpoolctl import threadpool_limits
    threadpool_limits(1)                 # same as the environment variables above, in case NumPy was already loaded
    torch.set_num_threads(max(1, N_CPU // 4) if DEVICE.type == "cpu" else 1)
import threading
READOUT_EPOCHS = 15            # authors' value (part of the replication, not a tuning knob)

def _ram_gb():
    try:
        with open("/proc/meminfo") as f:
            return round(int(f.readline().split()[1]) / 1e6, 1)
    except Exception:
        return None

ENV = {
    "python": platform.python_version(), "platform": platform.platform(),
    "torch": torch.__version__, "torch_pinned_2_4_0": TORCH_PINNED, "snntorch": snntorch.__version__,
    "sklearn": sklearn.__version__, "numpy": np.__version__, "scipy": scipy.__version__,
    "pandas": pd.__version__, "wfdb": wfdb.__version__, "device": str(DEVICE),
    "cpu_count": N_CPU, "ram_gb": _ram_gb(), "on_kaggle": ON_KAGGLE, "machine": MACHINE, "smoke_test": SMOKE_TEST,
    "pipeline": PIPELINE, "cpu_model": next((l.split(":", 1)[1].strip() for l in open("/proc/cpuinfo") if l.startswith("model name")), "?")
                 if os.path.exists("/proc/cpuinfo") else platform.processor(),
}
if DEVICE.type == "cuda":
    p = torch.cuda.get_device_properties(0)
    ENV.update({"gpu": p.name, "gpu_sm_count": p.multi_processor_count, "gpu_capability": f"{p.major}.{p.minor}",
                "gpu_mem_gb": round(p.total_memory / 1e9, 1), "cuda": torch.version.cuda})
try:
    import brian2
    ENV["brian2"] = brian2.__version__
except Exception as e:
    ENV["brian2"] = f"unavailable ({e.__class__.__name__})"

if SMOKE_TEST:
    SEEDS = SEEDS[:1]
    GAINS = [1, 21]
    DENSE_LAMBDAS = DENSE_LAMBDAS[:1]
    RETUNE_TRIALS = 2
    NN_EPOCHS = 1
    BRIAN2_BEATS = 2
    NN_TUNE_TRIALS = 2
    NN_TUNE_MAX_EPOCHS = 2
    NN_TUNE_CHECKPOINTS = [1, 2]
    HUANG_MATRICES = 2
    WORKING_DIAG_BEATS = 40
    SAVE_STATES = [s for s in SAVE_STATES if s["seed"] in SEEDS]
SMOKE_CONFIGS   = [11, 12, 13, 22]       # used only when SMOKE_TEST
SMOKE_CV_SPLITS = [(1, 1)]               # used only when SMOKE_TEST
SMOKE_PER_CLASS = {0: 10, 1: 5, 2: 5}    # beats kept per record and class when SMOKE_TEST

for k, v in ENV.items():
    print(f"{k:>20}: {v}")
if PIN_TORCH_240 and not TORCH_PINNED:
    print("NOTE: torch 2.4.0 is not active; the exact-replication comparison will still run but cannot match.")


## 2. The authors' code (pinned commit)

In [ ]:
# The authors' code, cloned at a pinned commit. Their functions are used directly;
# bugs are patched only inside the "fixed" liquid modes (see the liquid cell).
REPO_DIR = os.path.join(SCRATCH_DIR, "authors_repo")
os.makedirs(SCRATCH_DIR, exist_ok=True)
if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "clone", "-q", AUTHORS_REPO, REPO_DIR], check=True)
subprocess.run(["git", "-C", REPO_DIR, "checkout", "-q", AUTHORS_COMMIT], check=True)
ENV["authors_commit"] = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "HEAD"],
                                       capture_output=True, text=True).stdout.strip()
assert ENV["authors_commit"] == AUTHORS_COMMIT, ENV["authors_commit"]
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from src.preprocessing import signals as A_sig
from src.preprocessing import stratification as A_strat
from src.encoding import spike_coding as A_enc
from src.utils.reproducibility import seed_everything

def _load_script(name, rel):
    spec = importlib.util.spec_from_file_location(name, os.path.join(REPO_DIR, rel))
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)          # only imports + definitions; their main blocks are guarded
    return mod

A03 = _load_script("authors_cv_script", "scripts/03_run_24_configurations_cv.py")
LSM3D = A03.LiquidStateMachine3D           # identical class in scripts 02 and 03 (checked)

AUTHORS_CONFIGS = pd.read_csv(os.path.join(REPO_DIR, "configs/all_24_configurations.csv"))
ENC_SHORT = {"multithreshold": "multi", "temporal_rate": "rate", "bounded_delay_latency": "latency"}
CONFIGS = {int(r.config_id): {"config_id": int(r.config_id), "filter": r.filtering, "norm": r.normalization,
                              "encoder": ENC_SHORT[r.encoding], "readout": "svm" if r.readout_loss == "hinge" else "lr",
                              "authors_csv": r.source_result_file}
           for r in AUTHORS_CONFIGS.itertuples()}
print(f"authors' code at {ENV['authors_commit'][:10]}; {len(CONFIGS)} configurations loaded")


## 3. lsm2: the liquid code

In [ ]:
# lsm2 (the liquid code: config, liquids, simulator, statistics), written to SCRATCH_DIR and imported.
LSM2_FILES = {}
LSM2_FILES['lsm2/config.py'] = r'''"""Every fixed number of the redesign plan, with its source or reason.

Sources (as labelled in the plan):
  (a) Perez-Lopez et al. 2026 and the authors' code, commit 02e45c4
  (c) Maass, Natschlager & Markram 2002, Section 5 and Appendix B
  (d) Schmitt 2022, Brian2 reproduction of (c)'s Figure 2 (maass2002.py)
  (e) Legenstein & Maass 2007
Numbers marked "plan" are the plan's own choices; the reason is in the plan (section named).
Numbers marked "code" were not fixed by the plan; their reason is given here and listed in NOTES.md.
"""
import math

# ---------------------------------------------------------------- time
FS = 360.0                          # MIT-BIH sampling rate (Hz), (a)
SAMPLE_MS = 1000.0 / FS             # 2.7778 ms, one ECG sample = one step of (a)'s liquid
BEAT_SAMPLES = 577                  # (a): beats are R-0.8 s .. R+0.8 s at 360 Hz
SIM_SAMPLES = 650                   # (a): num_sim_steps = 650 (input is zero after the beat)
R_SAMPLE = 288                      # plan, Readouts: the R peak is sample 288 of the 577-sample window
WIN, NWIN = 20, 32                  # (a): spike counts in 32 windows of 20 samples (film readout)
SNAPSHOT_SAMPLES = (288, 342, 396)  # plan, Readouts: R, R + 150 ms, R + 300 ms
STATE_TAU_MS = 30.0                 # (c): liquid states are spikes filtered with a 30 ms exponential
N_SUB_REFERENCE = 28                # plan, Time grid: 28 sub-steps per sample (0.0992 ms) is the reference grid

# ---------------------------------------------------------------- X: (a)'s liquid, both bugs fixed
X = dict(
    betaE=0.95, betaI=0.80,         # (a) leak per 2.778 ms step (about 54 / 12 ms)
    threshold=1.0,                  # (a) Uthr
    rho=0.9,                        # (a) target fill of the paper's input rule
    excit_ratio=0.8,                # (a) and (c)
    C={"EE": 0.3, "EI": 0.2, "IE": 0.4, "II": 0.1},   # (a) and (c)
    lam=0.8,                        # (a)
    mu_rsv={"EE": 0.020, "EI": 0.025, "IE": 0.050, "II": 0.040},   # (a) type means
    P_in=0.1,                       # (a) code line 691
    dims=(7, 7, 7),                 # (a) 343 neurons
)
I_TARGET = X["rho"] * (1 - X["betaE"]) * X["threshold"]     # 0.045: (a)'s rule, mean input per step
TYPES = ("EE", "EI", "IE", "II")    # pre-post

# ---------------------------------------------------------------- Maass (c), in the paper's units (threshold 1)
MAASS = dict(
    tau_m_ms=30.0,                  # (c) membrane time constant
    v_thresh_mV=15.0, v_reset_mV=13.5, I_b_nA=13.5, R_Mohm=1.0,   # (c); background 13.5 mV of 15
    refrac_ms={"E": 3.0, "I": 2.0},  # (c)
    tau_s_ms={"E": 3.0, "I": 6.0},   # (c) synaptic current decay by presynaptic type
    tau_in_ms=3.0,                   # plan: input synapses are excitatory, so 3 ms for every target
    delay_ms={"EE": 1.5, "EI": 0.8, "IE": 0.8, "II": 0.8},   # (c)
    A_nA={"EE": 30.0, "EI": 60.0, "IE": -19.0, "II": -19.0},  # (c)
    U={"EE": 0.5, "EI": 0.05, "IE": 0.25, "II": 0.32},          # (c)
    D_s={"EE": 1.1, "EI": 0.125, "IE": 0.7, "II": 0.144},       # (c)
    F_s={"EE": 0.05, "EI": 1.2, "IE": 0.02, "II": 0.06},        # (c)
    udf_spread=0.5,                  # (c): SD 50% of the mean
    lam=2.0,                         # (c)
    in_nA={"E": 18.0, "I": 9.0},     # (c) input amplitudes
    in_fraction=0.3,                 # (c) input injected into 30% of neurons
    dims=(15, 3, 3),                 # (c) 135-neuron column
    input_rate_hz=20.0,              # (d) (and (c)'s Figure 4 task)
)
TAP = MAASS["I_b_nA"] * MAASS["R_Mohm"] / MAASS["v_thresh_mV"]    # 0.9 of threshold
RESET_TAP = MAASS["v_reset_mV"] / MAASS["v_thresh_mV"]            # 0.9

def _push(A_nA, tau_s_ms, U=1.0):
    """Voltage push (in thresholds) of one current pulse A*exp(-t/tau_s) through R into a 30 ms membrane:
    A * U * tau_s * R / (tau_m * V_thresh), the tap's convention (plan, factors 9 and 10)."""
    return A_nA * 1e-9 * U * tau_s_ms * 1e-3 * MAASS["R_Mohm"] * 1e6 / (MAASS["tau_m_ms"] * 1e-3 * MAASS["v_thresh_mV"] * 1e-3)

_pre = {"EE": "E", "EI": "E", "IE": "I", "II": "I"}
MAASS_PUSH = {t: abs(_push(MAASS["A_nA"][t], MAASS["tau_s_ms"][_pre[t]], MAASS["U"][t])) for t in TYPES}
# EE 0.1, EI 0.02, IE 0.0633, II 0.0811 (plan, factor 3: ratios 1 / 0.2 / 0.63 / 0.81; factor 10: EE 0.1)
MAASS_RATIOS = {t: MAASS_PUSH[t] / MAASS_PUSH["EE"] for t in TYPES}
MAASS_IN_PUSH = {k: _push(MAASS["in_nA"][k], MAASS["tau_in_ms"]) for k in ("E", "I")}   # 0.12 / 0.06 (factor 9)
MAASS_EE_PUSH = MAASS_PUSH["EE"]   # 0.1 (factor 10)

# ---------------------------------------------------------------- adaptive thresholds (Bellec et al. 2018, LSNN)
# Bellec, Salaj, Subramoney, Legenstein & Maass 2018, "Long short-term memory and learning-to-learn in networks of
# spiking neurons", Section 5 parameters: threshold B_j = b0 + beta b_j, b_j <- rho b_j + (1 - rho) z_j with
# rho = exp(-dt / tau_a) and z_j = 1/dt (ms) at a spike; beta 1.6, b0 0.03, tau_a uniform on 1..1000 ms, 40% of the
# neurons adaptive. Translation to Maass's neurons (rest 0.9, threshold 1): the threshold rise is taken relative to the
# rest-to-threshold gap, theta_j = 1 + (1 - 0.9) beta b_j / b0, since that gap decides firing in both models.
ADAPT = dict(beta=1.6, b0=0.03, tau_ms=(1.0, 1000.0), fraction=0.4)

# ---------------------------------------------------------------- the 11 factors (plan table)
FACTOR_NAMES = {
    1: "reach lambda", 2: "code-only normalisations", 3: "E/I balance", 4: "leak", 5: "operating point",
    6: "input wiring", 7: "fine timing", 8: "dynamic synapses", 9: "input size", 10: "weight scale",
    11: "size and shape",
}
ALL_FACTORS = frozenset(range(1, 12))
LEAK_MS_Y = 30.0                    # factor 4: (c)'s 30 ms for E and I

# ---------------------------------------------------------------- gain grid (plan, Choosing the recurrent gain; decision 5)
G_LOW = (0.1, 0.3)
G_COARSE = (1, 3, 10, 30, 100, 300, 1000)
G_EXTEND_STEP, G_EXTEND_MAX = 3.0, 3
G_FINE_STEP, G_FINE_K = 1.4, 3      # g* x 1.4^k, k = -3..3 (1.4^3 ~ 2.7 splits a 3x step into three)

# ---------------------------------------------------------------- level thresholds (plan, Correctness criteria)
TOL_1A = 1e-9                       # 1a, 1b open loop (float64)
TOL_1C = 1e-4                       # 1c float32 vs float64
LEVEL2_SE = 4.5                     # level 2 (decision 24)
BAND_HZ = (2.0, 40.0)               # level 3 firing band (decision 7)
HIGH_HZ, HIGH_FRAC = 100.0, 0.05    # at most 5% of neurons above 100 Hz (decision 7)
N_CAL_BEATS = 500                   # plan: 500 beats for level 3 and the rank scores (decision 8)
GEN_BEATS, GEN_COPIES = 4, 125      # decision 8: 4 beats x 125 jittered copies
JITTER_MS = 4.0                     # decision 8: (e)'s 4 ms jitter for its 30-channel patterns
STATE_READ_SAMPLE = 396             # decision 8: states read at R + 300 ms
START_TEST_SAMPLE = 576             # level 3: starting-state rule judged at sample 576
MEMORY_T_MS = (0, 25, 50, 100, 200, 400)   # level 4 lags
MEMORY_PASS_T_MS = 100              # level 4 pass lag (plan: well past 30-54 ms membranes and 3-6 ms currents)
MEMORY_MIN_GAIN = 0.05              # level 4 (decision 9)
MEMORY_WARMUP = 100                 # level 4: 278 ms warm-up
SURROGATE_TOL = 0.10                # level 4: surrogate spikes per sample within 10% of ECG's
E1D_N_TRY_MAX = 28                  # 1d: n = 1, 2, 3, ... up to the reference grid
E1E_T_CRIT = 4.2                    # 1e: t at 8 df, two-sided 0.3%
E1E_FRAC = 0.95                     # 1e: 95% of time points
E1E_RUNS = 5                        # 1e: 5 runs per side

SEEDS_SEARCH = (1, 2, 3)            # code: liquid seeds for the ladder search (the plan fixes 3 seeds, not their values)
SEEDS_CONFIRM = (101, 202, 303)     # plan, confirmation
SEEDS_RETRY = (404, 505, 606)       # plan, confirmation retry
SEEDS_TEST_REDESIGN = (707, 808, 909)   # plan, Maass check redesign
SEEDS_CLASSIFY = (2026, 69, 420)    # plan, ground rules

# ---------------------------------------------------------------- code choices the plan does not fix (see NOTES.md)
CODE = dict(
    twin_beats=100,          # twin-run cross-check per g and seed: a reported cross-check, 100 beats give a stable mean ratio
    twin_early=(289, 306),   # samples of the first 50 ms after the extra spike
    twin_late=(559, 576),    # samples of the last 50 ms of the beat
    memory_sequences=500,    # level 4: same count as level 3's beats; 250 train / 250 test sequences
    ridge_alphas=tuple(10.0 ** k for k in range(-3, 5)),   # level 4 inner-CV penalty grid
    ridge_inner_folds=5,     # level 4 inner CV (grouped by sequence)
    calib_tol=0.01,          # factor 5: bisection stops at 1% of the target rate (well inside seed-to-seed spread)
    calib_iters=40,          # factor 5: bisection cap
    search_split_seed=0,     # DS1 patients halved by a fixed random permutation of the 22 records
    sim_batch=500,           # beats per GPU batch (fixed on every machine: plan, determinism)
    x_update="equations",    # X's reset follows (a)'s Eqs. 17-19 (the plan). "authors_code" (the snnTorch neuron of
                             # (a)'s code) exists only so the tests can show the difference; the notebook never sets it
)
'''
LSM2_FILES['lsm2/liquid.py'] = r'''"""Building liquids: every ladder candidate (X plus any set of the 11 factors), Maass's two versions for the
Maass check (script and paper), and X converted from the authors' own class (for 1f and E1's paper liquid).

A Liquid holds the realised network and every time constant already converted to its grid (n sub-steps per
input step). Input weights are stored as a base matrix; the input size (paper's rule, factor-5 calibration or
factor 9) is a separate scalar `in_scale`, and the recurrent gain g a scalar given to the simulator.
"""
from dataclasses import dataclass, field
import math
import numpy as np

from . import config as C

TYPE_INDEX = {t: i for i, t in enumerate(C.TYPES)}     # EE 0, EI 1, IE 2, II 3


def round_half_up(x):
    return int(math.floor(x + 0.5))


def steps_of(ms, h_ms):
    """A time in ms as a whole number of sub-steps: rounded to the nearest, at least 1 (plan, Time grid)."""
    return max(1, int(round(ms / h_ms)))


@dataclass
class Liquid:
    # structure
    N: int
    dims: tuple
    is_exc: np.ndarray          # (N,) bool
    pos: np.ndarray             # (N, 3)
    pre: np.ndarray             # (S,) int
    post: np.ndarray            # (S,) int
    typ: np.ndarray             # (S,) int 0..3 (EE, EI, IE, II)
    w: np.ndarray               # (S,) signed base weight: a push in thresholds ("ours"), or J amplitude ("brian2")
    U: np.ndarray               # (S,) utilisation (dynamic synapses), else ones
    D_ms: np.ndarray            # (S,)
    F_ms: np.ndarray            # (S,)
    Ubar: np.ndarray            # (4,) nominal type means of U
    Win: np.ndarray             # (C, N) base input weights, >= 0
    in_neurons: np.ndarray      # (N,) bool: neurons with any input synapse
    # time grid and neuron model
    n_sub: int                  # sub-steps per input step
    h_ms: float                 # sub-step length
    beta: np.ndarray            # (N,) leak per sub-step
    tap: np.ndarray             # (N,) tap per sub-step ("ours"); 0 without factor 5
    reset_mode: str             # "subtract" or "value"
    reset_value: float
    start_lo: float             # starting voltages drawn uniformly in [start_lo, start_hi]; equal = fixed
    start_hi: float
    rest: float                 # resting level (0, or 0.9 with the tap): lower end of level 3's random starts
    gap: np.ndarray             # (N,) int minimum gap between spikes, in sub-steps
    clamp: bool                 # voltage clamped during the blocked sub-steps
    strict: bool                # threshold test u > 1 (Brian2 script) instead of u >= 1 (ours, (a)'s H)
    schedule: str               # "ours" or "brian2"
    syn_model: str              # "static", "markram" (two-state, (c)'s form) or "tsodyks3" ((d)'s three-state)
    delay: np.ndarray           # (4,) int sub-steps by type
    cur_a: object               # None (push at once) or (2,) decay per sub-step for E- and I-presynaptic currents
    in_a: object                # None or (N,) decay per sub-step of each neuron's input current
    # Brian2-schedule constants (script version only)
    J_b: float = 0.0
    kappa: object = None        # (N,) exact weight of the decaying input current in the membrane update
    syn_tau_ms: object = None   # (4,) current decay per type (tsodyks3)
    # bookkeeping
    factors: frozenset = frozenset()
    snn_quirk: bool = False     # (a)'s code (snnTorch Leaky, reset_delay=False): the threshold test subtracts 1 more
                                # when the stored voltage is above 1; u > 1 strict. Default: (a)'s Eqs. 17-19
    g_maass: float = None       # g at Maass's own scale (factor 10)
    in_scale_fixed: float = None   # input size fixed by the design (factor 9, Maass versions)
    in_split: tuple = (1.0, 1.0)   # base input means onto E / I neurons (2:1 with factors 5 or 9)
    adapt_c: object = None         # (N,) threshold rise per unit of b (0 = not adaptive); None = no adaptation
    adapt_tau_ms: object = None    # (N,) adaptation time constants (ms)
    meta: dict = field(default_factory=dict)

    @property
    def S(self):
        return len(self.pre)

    def ee_mean_push(self, g=1.0):
        """Mean isolated EE push per presynaptic spike, in thresholds (the plan's absolute g scale)."""
        m = self.typ == 0
        if not m.any():
            return 0.0
        iso = np.abs(self.w[m]) * (self.U[m] / self.Ubar[0] if self.syn_model == "markram" else 1.0)
        return float(g * iso.mean())


# ---------------------------------------------------------------- shared pieces
def _grid(dims):
    return np.stack(np.meshgrid(*[np.arange(d) for d in dims], indexing="ij"), -1).reshape(-1, 3).astype(np.float64)


def _type_matrix(is_exc):
    pe, qe = is_exc[:, None], is_exc[None, :]
    T = np.where(pe & qe, 0, np.where(pe & ~qe, 1, np.where(~pe & qe, 2, 3)))
    return T


def _draw_udf(rng_n, rng_u, T, mean_of_type, cap, bad_rule):
    """U, D or F for every (pre, post) pair: Normal(mean, 0.5 mean); draws at or below 0 replaced by
    uniform(0, 2 mean) ("redraw", plan decision 17) or by the mean ("mean", (d)); U capped at 1 if cap."""
    mean = np.vectorize(lambda t: mean_of_type[C.TYPES[t]])(T).astype(np.float64)
    z = rng_n.standard_normal(T.shape)
    v = mean + C.MAASS["udf_spread"] * mean * z
    repl = rng_u.random(T.shape) * 2 * mean
    if bad_rule == "redraw":
        v = np.where(v <= 0, repl, v)
    else:      # (d): U < 0 -> mean; D, F <= 0 -> mean
        v = np.where(v <= 0, mean, v)
    if cap:
        v = np.minimum(v, 1.0)
    return v


# ---------------------------------------------------------------- ladder candidates
def candidate_settings(factors):
    f = frozenset(factors)
    return {
        "dims": C.MAASS["dims"] if 11 in f else C.X["dims"],
        "lam": C.MAASS["lam"] if 1 in f else C.X["lam"],
        "mu": ({t: C.X["mu_rsv"]["EE"] * C.MAASS_RATIOS[t] for t in C.TYPES} if 3 in f else dict(C.X["mu_rsv"])),
        "normalise": 2 not in f,
        "tau_E_ms": C.LEAK_MS_Y if 4 in f else -C.SAMPLE_MS / math.log(C.X["betaE"]),
        "tau_I_ms": C.LEAK_MS_Y if 4 in f else -C.SAMPLE_MS / math.log(C.X["betaI"]),
        "tap": 5 in f,
        "input30": 6 in f,
        "fine": 7 in f,
        "dynamic": 8 in f,
        "maass_input": 9 in f,
        "maass_scale": 10 in f,
    }


def build_candidate(factors, seed, n_sub, n_channels, n_sub_input_step_ms=C.SAMPLE_MS, in_fraction=None):
    """A ladder candidate: X ((a), bugs fixed) with the given factors switched on.
    n_sub: sub-steps per sample (1 without factor 7 unless a grid check asks otherwise).
    in_fraction (factor 6 only): share of neurons that receive the input; None = (c)'s 30%. 1.0 = every neuron, as
    in (d)'s script. The input weights come from the same draw either way, so the 30% neurons keep their weights."""
    f = frozenset(factors)
    s = candidate_settings(f)
    N = int(np.prod(s["dims"]))
    h = n_sub_input_step_ms / n_sub
    # independent streams, so candidates with the same seed share their random numbers where they can
    r_types, r_conn, r_w = (np.random.default_rng([seed, k]) for k in (1, 2, 3))
    r_un, r_uu, r_dn, r_du, r_fn, r_fu = (np.random.default_rng([seed, k]) for k in (4, 5, 6, 7, 8, 9))
    r_inm, r_inw, r_inn = (np.random.default_rng([seed, k]) for k in (10, 11, 12))
    n_exc = int(N * C.X["excit_ratio"])
    is_exc = np.zeros(N, bool)
    is_exc[r_types.permutation(N)[:n_exc]] = True
    pos = _grid(s["dims"])
    T = _type_matrix(is_exc)
    Cm = np.vectorize(lambda t: C.X["C"][C.TYPES[t]])(T)
    d = np.sqrt(((pos[:, None, :] - pos[None, :, :]) ** 2).sum(-1))
    p = np.clip(Cm * np.exp(-(d / s["lam"]) ** 2), 0, 1)
    conn = r_conn.random((N, N)) < p
    np.fill_diagonal(conn, False)                            # no self-connections ((a); kept in Y)
    mu = np.vectorize(lambda t: s["mu"][C.TYPES[t]])(T)
    Wmag = r_w.standard_exponential((N, N)) * mu             # Gamma(shape 1, scale mu): spread = mean
    if s["normalise"]:
        fan_in = np.maximum(conn.sum(0, keepdims=True), 1)   # (a) lines 168-169: divide by the receiver's count
        Wmag = Wmag / fan_in
    W = np.where(is_exc[:, None], Wmag, -Wmag)
    pre, post = np.nonzero(conn)
    order = np.lexsort((post, pre, T[pre, post]))           # sorted by type, then pre, then post
    pre, post = pre[order], post[order]
    typ = T[pre, post]
    if s["dynamic"]:
        Ufull = _draw_udf(r_un, r_uu, T, C.MAASS["U"], True, "redraw")
        Dfull = _draw_udf(r_dn, r_du, T, {k: v * 1000 for k, v in C.MAASS["D_s"].items()}, False, "redraw")
        Ffull = _draw_udf(r_fn, r_fu, T, {k: v * 1000 for k, v in C.MAASS["F_s"].items()}, False, "redraw")
        U, D, F = Ufull[pre, post], Dfull[pre, post], Ffull[pre, post]
    else:
        U, D, F = np.ones(len(pre)), np.full(len(pre), np.inf), np.full(len(pre), np.inf)
    Ubar = np.array([C.MAASS["U"][t] for t in C.TYPES])
    # input wiring and base weights
    if s["input30"]:
        k = round_half_up((C.MAASS["in_fraction"] if in_fraction is None else in_fraction) * N)
        in_n = np.zeros(N, bool)
        in_n[r_inn.permutation(N)[:k]] = True
        mask = np.repeat(in_n[None, :], n_channels, 0)
    else:
        mask = r_inm.random((n_channels, N)) < C.X["P_in"]
        in_n = mask.any(0)
    split = (1.0, 0.5) if (s["tap"] or s["maass_input"]) else (1.0, 1.0)
    m_in = np.where(is_exc, split[0], split[1])[None, :]
    Win = r_inw.standard_exponential((n_channels, N)) * m_in * mask
    if s["normalise"]:                                       # (a) lines 71-75: each column's mean active weight = m
        cnt = mask.sum(0, keepdims=True)
        col = Win.sum(0, keepdims=True) / (cnt + 1e-8)
        Win = Win * (m_in / (col + 1e-8))
    # time constants on the grid
    tau = np.where(is_exc, s["tau_E_ms"], s["tau_I_ms"])
    beta = np.exp(-h / tau)
    tap = C.TAP * (1 - beta) if s["tap"] else np.zeros(N)
    if s["fine"]:
        gap = np.where(is_exc, steps_of(C.MAASS["refrac_ms"]["E"], h), steps_of(C.MAASS["refrac_ms"]["I"], h))
        delay = np.array([steps_of(C.MAASS["delay_ms"][t], h) for t in C.TYPES])
        cur_a = np.exp(-h / np.array([C.MAASS["tau_s_ms"]["E"], C.MAASS["tau_s_ms"]["I"]]))
        in_a = np.full(N, math.exp(-h / C.MAASS["tau_in_ms"]))
        clamp = True
    else:
        # X: the 2.78 ms gap and the one-sample delay, in ms, converted to the grid in use
        gap = np.full(N, steps_of(C.SAMPLE_MS, h))
        delay = np.full(4, steps_of(C.SAMPLE_MS, h))
        cur_a, in_a, clamp = None, None, False
    liq = Liquid(
        N=N, dims=tuple(s["dims"]), is_exc=is_exc, pos=pos, pre=pre, post=post, typ=typ, w=W[pre, post].astype(np.float64),
        U=U, D_ms=D, F_ms=F, Ubar=Ubar, Win=Win.astype(np.float64), in_neurons=in_n,
        n_sub=n_sub, h_ms=h, beta=beta, tap=tap,
        reset_mode="value" if s["tap"] else "subtract", reset_value=C.RESET_TAP if s["tap"] else 0.0,
        start_lo=C.RESET_TAP if s["tap"] else 0.0, start_hi=1.0 if s["tap"] else 0.0,
        rest=C.TAP if s["tap"] else 0.0,
        gap=gap.astype(np.int64), clamp=clamp, strict=False, schedule="ours",
        syn_model="markram" if s["dynamic"] else "static", delay=delay.astype(np.int64), cur_a=cur_a, in_a=in_a,
        factors=f, in_split=split,
        meta={"kind": "candidate", "seed": int(seed), "n_channels": int(n_channels), "settings": s},
    )
    if liq.reset_mode == "subtract" and C.CODE["x_update"] == "authors_code":
        liq.snn_quirk, liq.strict = True, True
    m = typ == 0
    liq.g_maass = C.MAASS_EE_PUSH / float(np.abs(liq.w[m]).mean()) if m.any() else None
    if s["maass_input"]:
        liq.in_scale_fixed = C.MAASS_IN_PUSH["E"]           # base means 1 / 0.5 -> 0.12 / 0.06 per input spike
    return liq


def add_adaptation(liq, seed, beta=None, b0=None, tau_ms=None, fraction=None):
    """Adaptive thresholds (config.ADAPT, Bellec et al. 2018 Section 5) on a random `fraction` of the neurons, each
    with its own tau_a drawn uniformly from tau_ms. Own random streams, so the rest of the liquid is unchanged."""
    a = C.ADAPT
    beta = a["beta"] if beta is None else beta
    b0 = a["b0"] if b0 is None else b0
    lo, hi = a["tau_ms"] if tau_ms is None else tau_ms
    frac = a["fraction"] if fraction is None else fraction
    N = liq.N
    k = round_half_up(frac * N)
    chosen = np.zeros(N, bool)
    chosen[np.random.default_rng([seed, 13]).permutation(N)[:k]] = True
    tau = np.full(N, np.inf)
    tau[chosen] = np.random.default_rng([seed, 14]).uniform(lo, hi, k)
    gap = 1.0 - (liq.rest if liq.rest > 0 else 0.0)
    liq.adapt_c = np.where(chosen, gap * beta / b0, 0.0)
    liq.adapt_tau_ms = tau
    liq.meta["adaptation"] = {"beta": beta, "b0": b0, "tau_ms": [lo, hi], "fraction": frac, "n_adaptive": int(k), "gap": gap}
    return liq


def p_in_effective(liq):
    """Connection probability the paper's rule uses: P_in = 0.1, or 1 for the input neurons with factor 6."""
    return 1.0 if 6 in liq.factors else C.X["P_in"]


def paper_rule_scale(ain, liq):
    """(a)'s rule: mu_in = I_target / (A_in P_in), always with X's excitatory leak (plan, Input size)."""
    return C.I_TARGET / (ain * p_in_effective(liq))


# ---------------------------------------------------------------- Maass's liquid, two versions (Maass check)
def build_maass_paper(seed, n_sub=None, n_channels=1):
    """Our reading of (c), the version Y copies: Y with all 11 factors and one Poisson input channel."""
    n_sub = n_sub or C.N_SUB_REFERENCE
    liq = build_candidate(C.ALL_FACTORS, seed, n_sub, n_channels)
    liq.meta["kind"] = "maass_paper"
    return liq


def build_maass_script(seed, n_sub=None, step_ms=C.SAMPLE_MS, d_replacement="fixed"):
    """The liquid of (d)'s Brian2 script, in threshold units, with its six settings (plan, level 1e):
    input onto I neurons decays over 6 ms, bad U, D, F draws replaced by the mean, U not capped, no input delay,
    an arriving spike acts after that step's state update (Brian2 schedule), each step's synaptic current held
    constant (exact neuron update with step-constant synaptic current). Also from (d): input to all 135 neurons
    at a fixed 18 / 9 nA, self-connections allowed, E neurons are the first 108 indices with shuffled positions,
    three-state synapses (used resources recover only once the current has ended)."""
    n_sub = n_sub or C.N_SUB_REFERENCE
    N = int(np.prod(C.MAASS["dims"]))
    h = step_ms / n_sub
    rng = np.random.default_rng([seed, 1000])
    idx = rng.permutation(N)
    pos = np.stack([idx % 3, (idx // 3) % 3, idx // 9], 1).astype(np.float64)
    n_exc = int(0.8 * N)
    is_exc = np.zeros(N, bool); is_exc[:n_exc] = True
    T = _type_matrix(is_exc)
    Cm = np.vectorize(lambda t: C.X["C"][C.TYPES[t]])(T)
    d2 = ((pos[:, None, :] - pos[None, :, :]) ** 2).sum(-1)
    p = Cm * np.exp(-d2 / C.MAASS["lam"] ** 2)
    conn = np.random.default_rng([seed, 1001]).random((N, N)) < p          # self-connections allowed ((d))
    pre, post = np.nonzero(conn)
    order = np.lexsort((post, pre, T[pre, post]))
    pre, post = pre[order], post[order]
    typ = T[pre, post]
    J = C.MAASS["R_Mohm"] / C.MAASS["v_thresh_mV"]                         # nA -> thresholds (1 MOhm / 15 mV)
    A = np.array([C.MAASS["A_nA"][C.TYPES[t]] for t in typ])
    r = np.random.default_rng([seed, 1002])
    w = np.sign(A) * r.standard_exponential(len(pre)) * np.abs(A) * J       # Gamma(1, |A|) with sign ((d))
    def udf(name, scale, k):
        mean = np.array([C.MAASS[name][C.TYPES[t]] for t in typ]) * scale
        v = mean + 0.5 * mean * np.random.default_rng([seed, k]).standard_normal(len(pre))
        if d_replacement == "as_written":       # (d) as written: only the first two synapses of each group
            for ti in range(4):
                first = np.nonzero(typ == ti)[0][:2]
                v[first] = mean[first]
            return v
        return np.where(v < 0, mean, v) if name == "U" else np.where(v <= 0, mean, v)
    U, D, F = udf("U", 1.0, 1003), udf("D_s", 1000.0, 1004), udf("F_s", 1000.0, 1005)
    Ubar = np.array([C.MAASS["U"][t] for t in C.TYPES])
    Win = (np.where(is_exc, C.MAASS["in_nA"]["E"], C.MAASS["in_nA"]["I"]) * J)[None, :]   # J jump per input spike
    tau_m = C.MAASS["tau_m_ms"]
    beta = np.full(N, math.exp(-h / tau_m))
    in_tau = np.where(is_exc, 3.0, 6.0)                                     # (d): tau_stimulus 3 ms (E), 6 ms (I)
    a_in = np.exp(-h / in_tau)
    kappa = in_tau / (in_tau - tau_m) * (np.exp(-h / in_tau) - np.exp(-h / tau_m))   # Brian2's exact solution
    gap = np.where(is_exc, int((C.MAASS["refrac_ms"]["E"] + 1e-3 * h) / h), int((C.MAASS["refrac_ms"]["I"] + 1e-3 * h) / h))
    delay = np.array([int(round(C.MAASS["delay_ms"][t] / h)) for t in C.TYPES])   # Brian2 rounds delays to the grid
    syn_tau = np.array([C.MAASS["tau_s_ms"]["E"], C.MAASS["tau_s_ms"]["E"], C.MAASS["tau_s_ms"]["I"], C.MAASS["tau_s_ms"]["I"]])
    liq = Liquid(
        N=N, dims=C.MAASS["dims"], is_exc=is_exc, pos=pos, pre=pre, post=post, typ=typ, w=w, U=U, D_ms=D, F_ms=F,
        Ubar=Ubar, Win=Win, in_neurons=np.ones(N, bool), n_sub=n_sub, h_ms=h, beta=beta, tap=np.zeros(N),
        reset_mode="value", reset_value=C.RESET_TAP, start_lo=C.RESET_TAP, start_hi=1.0, rest=C.TAP,
        gap=gap.astype(np.int64), clamp=True, strict=True, schedule="brian2", syn_model="tsodyks3",
        delay=delay.astype(np.int64), cur_a=None, in_a=a_in, J_b=C.TAP, kappa=kappa, syn_tau_ms=syn_tau,
        factors=frozenset(), in_scale_fixed=1.0, in_split=(1.0, 0.5),
        meta={"kind": "maass_script", "seed": int(seed), "n_channels": 1, "d_replacement": d_replacement},
    )
    liq.g_maass = 1.0
    return liq


# ---------------------------------------------------------------- X from the authors' class
def from_authors_model(model, n_sub=1, snn_quirk=None):
    """X exactly as (a)'s class built it (weights already include its normalisations and its g_rec):
    used for the 1f regression and for E1's paper liquid at g = 1. Input weights are returned at their size
    (in_scale 1)."""
    import torch
    W = model.rsv_weights.detach().double().cpu().numpy()
    Win = model.input_rsv_weights.detach().double().cpu().numpy()
    is_exc = model.is_excit.detach().cpu().numpy().astype(bool)
    N = W.shape[0]
    T = _type_matrix(is_exc)
    pre, post = np.nonzero(W)
    order = np.lexsort((post, pre, T[pre, post]))
    pre, post = pre[order], post[order]
    h = C.SAMPLE_MS / n_sub
    tau = np.where(is_exc, -C.SAMPLE_MS / math.log(C.X["betaE"]), -C.SAMPLE_MS / math.log(C.X["betaI"]))
    liq = Liquid(
        N=N, dims=tuple(model.reservoir_dim), is_exc=is_exc, pos=_grid(model.reservoir_dim), pre=pre, post=post,
        typ=T[pre, post], w=W[pre, post], U=np.ones(len(pre)), D_ms=np.full(len(pre), np.inf),
        F_ms=np.full(len(pre), np.inf), Ubar=np.array([C.MAASS["U"][t] for t in C.TYPES]), Win=Win,
        in_neurons=(Win != 0).any(0), n_sub=n_sub, h_ms=h, beta=np.exp(-h / tau), tap=np.zeros(N),
        reset_mode="subtract", reset_value=0.0, start_lo=0.0, start_hi=0.0, rest=0.0,
        gap=np.full(N, steps_of(C.SAMPLE_MS, h)), clamp=False, strict=False, schedule="ours", syn_model="static",
        delay=np.full(4, steps_of(C.SAMPLE_MS, h)), cur_a=None, in_a=None, factors=frozenset(),
        in_scale_fixed=1.0, meta={"kind": "authors_X", "n_channels": int(Win.shape[0])},
    )
    if snn_quirk or (snn_quirk is None and C.CODE["x_update"] == "authors_code"):
        liq.snn_quirk, liq.strict = True, True
    return liq
'''
LSM2_FILES['lsm2/sim.py'] = r'''"""The simulator: one batched update for X, Y and every candidate between them (each factor is a setting of the
Liquid), plus the Brian2 schedule used only for (d)'s own liquid (level 1e and the Maass check's script version).

Units: thresholds (the voltage threshold is 1). Time: input steps (one ECG sample, or 0.1 ms for 1e), each split
into n sub-steps. Our order within a sub-step (plan, Simulation details):
  1. synapse states recover/fade over the sub-step; spikes whose delay ends now arrive (dynamic synapses
     update u, push, R in (d)'s order)
  2. currents: without fine timing a push arrives all at once; with it, it feeds a current that decays over
     3 ms (from E and from inputs) or 6 ms (from I), scaled so its total equals the push
  3. every neuron integrates: u <- beta u + input + recurrent + tap (not during a clamped refractory period)
  4. threshold test (u >= 1, outside the minimum gap), then reset (subtract 1, or set to 0.9 with the tap)
Input spikes of an input step act in its first sub-step ((a)'s Eq. 16: x[t] W_in in step t); this step's spikes
reach their targets from the next sub-step on (delay >= 1).
"""
import math
import numpy as np
import torch

from . import config as C

# Fast path (same arithmetic, fewer Python round trips): the "ours" schedule run one input step at a time on fixed
# buffers, and on a CUDA card each input step replayed as one captured CUDA graph. The notebook compares it with the
# plain loop on the card before using it and switches it off if they differ.
FAST = True
USE_CUDA_GRAPHS = True


def _t(a, device, dtype):
    return torch.as_tensor(np.asarray(a), device=device, dtype=dtype)


class Simulator:
    def __init__(self, liq, device="cpu", dtype=torch.float64, g=1.0, in_scale=None, recurrence=True):
        self.liq, self.device, self.dtype = liq, torch.device(device), dtype
        self.g = float(g) if recurrence else 0.0
        if in_scale is None:
            if liq.in_scale_fixed is None:
                raise ValueError("this liquid needs an input size (paper rule or calibration)")
            in_scale = liq.in_scale_fixed
        self.in_scale = float(in_scale)
        d, t = self.device, dtype
        N = liq.N
        self.N = N
        self.beta = _t(liq.beta, d, t)
        self.tap = _t(liq.tap, d, t)
        self.gap = _t(liq.gap, d, torch.int64)
        self.Win = _t(liq.Win * self.in_scale, d, t)
        self.z_decay = math.exp(-liq.h_ms / C.STATE_TAU_MS)
        # adaptive thresholds: theta = 1 + c b; b <- rho b + (1 - rho) spike / h  (b in spikes per ms)
        self.adapt = liq.adapt_c is not None and bool(np.any(np.asarray(liq.adapt_c) > 0))
        if self.adapt:
            self.a_c = _t(liq.adapt_c, d, t)
            rho = np.exp(-liq.h_ms / np.asarray(liq.adapt_tau_ms, float))     # 1 for non-adaptive neurons (tau inf)
            self.a_rho = _t(rho, d, t)
            self.a_in = _t((1.0 - rho) / liq.h_ms, d, t)
        self.L = int(max(liq.delay.max(), 1)) + 1           # spike-history ring length
        S = liq.S
        self.S = S
        self.static = liq.syn_model == "static"
        if liq.schedule == "ours":
            if self.static:
                # one dense matrix per (delay, presynaptic class): pushes of the spikes that left `delay` sub-steps ago
                self.groups = {}
                for ti in range(4):
                    m = liq.typ == ti
                    if not m.any():
                        continue
                    key = (int(liq.delay[ti]), 0 if ti < 2 else 1)
                    W = self.groups.get(key)
                    if W is None:
                        W = torch.zeros(N, N, device=d, dtype=t)
                    W[_t(liq.pre[m], d, torch.int64), _t(liq.post[m], d, torch.int64)] += _t(liq.w[m] * self.g, d, t)
                    self.groups[key] = W
            else:
                self._prep_sparse()
                Ubar_s = liq.Ubar[liq.typ]
                self.wg = _t(liq.w * self.g / Ubar_s, d, t)        # push = (w / U-bar) u R
                self.U = _t(liq.U, d, t)
                self.eD = _t(np.exp(-liq.h_ms / liq.D_ms), d, t)
                self.eF = _t(np.exp(-liq.h_ms / liq.F_ms), d, t)
            if liq.cur_a is not None:
                self.aE, self.aI = float(liq.cur_a[0]), float(liq.cur_a[1])
            self.in_a = None if liq.in_a is None else _t(liq.in_a, d, t)
        else:   # Brian2 schedule, (d)'s three-state synapses, currents in voltage-equivalent units
            self._prep_sparse()
            self.wg = _t(liq.w * self.g, d, t)
            self.U = _t(liq.U, d, t)
            tau = liq.syn_tau_ms[liq.typ]
            D = liq.D_ms
            h = liq.h_ms
            self.ey = _t(np.exp(-h / tau), d, t)
            self.eD = _t(np.exp(-h / D), d, t)
            self.cxy = _t(tau * (np.exp(-h / tau) - np.exp(-h / D)) / (D - tau), d, t)
            self.eF = _t(np.exp(-h / liq.F_ms), d, t)
            self.kappa = _t(liq.kappa, d, t)
            self.in_a = _t(liq.in_a, d, t)
            self.J_b = float(liq.J_b)

    def _prep_sparse(self):
        liq, d, t = self.liq, self.device, self.dtype
        self.pre_idx = _t(liq.pre, d, torch.int64)
        self.syn_delay = liq.delay[liq.typ]
        self.delay_groups = [(int(dl), _t(np.nonzero(self.syn_delay == dl)[0], d, torch.int64))
                             for dl in np.unique(self.syn_delay)]
        self.post_idx = _t(liq.post, d, torch.int64)
        self.nE = int((liq.typ < 2).sum())                  # synapses are sorted by type: E-presynaptic first
        M = torch.zeros(liq.S, liq.N, device=d, dtype=t)
        if liq.S:
            M[torch.arange(liq.S, device=d), _t(liq.post, d, torch.int64)] = 1.0
        self.M_E, self.M_I, self.M = M[:self.nE], M[self.nE:], M

    # ------------------------------------------------------------------ helpers
    def _arrivals(self, hist, k, B):
        """(B, S) 0/1: which synapses receive a spike in sub-step k."""
        a = torch.zeros(B, self.S, device=self.device, dtype=self.dtype)
        for dl, idx in self.delay_groups:
            if k - dl < 0:
                continue
            src = hist[(k - dl) % self.L]
            a[:, idx] = src[:, self.pre_idx[idx]]
        return a

    def start_state(self, u0):
        return torch.as_tensor(u0, device=self.device, dtype=self.dtype).clone()

    # ------------------------------------------------------------------ main loop
    @torch.no_grad()
    def run(self, x, u0, T, rec=None, open_loop=None, trace=False, own_spikes=None, fast=None):
        """x: (B, T_in, C) input spikes per input step; u0: (B, N) starting voltages; T: input steps to simulate.
        rec: {"windows": (win, nwin), "states_at": [t, ...], "states_every": (t0, t1), "neuron_counts": (t0, t1),
              "raster": bool}; times are input steps, a state is read at the end of its step.
        open_loop: (T*n, B, N) recorded spikes used as the presynaptic spikes (the liquid's own spikes are still
        computed and returned). own_spikes: (T*n, B, N) recorded spikes that also replace each neuron's own
        threshold decisions (level 1c: pure arithmetic drift, no flips); out["flips"] counts where the threshold
        test would have decided differently. trace: return u and currents every sub-step and synapse states
        every input step. fast: None = the module's FAST; False = this plain loop (the reference)."""
        liq, dev, dt = self.liq, self.device, self.dtype
        rec = rec or {}
        x = torch.as_tensor(x, device=dev)
        if (FAST if fast is None else fast) and liq.schedule == "ours" and open_loop is None and own_spikes is None and not trace \
                and not rec.get("raster"):
            return self._run_fast(x, u0, T, rec, graphs=USE_CUDA_GRAPHS and dev.type == "cuda")
        B, T_in, Cin = x.shape
        N, n = self.N, liq.n_sub
        u = self.start_state(u0)
        ref = torch.zeros(B, N, device=dev, dtype=torch.int64)
        hist = torch.zeros(self.L, B, N, device=dev, dtype=dt)
        z = torch.zeros(B, N, device=dev, dtype=dt)
        ab = torch.zeros(B, N, device=dev, dtype=dt) if self.adapt else None
        ours = liq.schedule == "ours"
        fine = ours and liq.cur_a is not None
        IE = torch.zeros(B, N, device=dev, dtype=dt) if fine else None
        II = torch.zeros(B, N, device=dev, dtype=dt) if fine else None
        Iin = torch.zeros(B, N, device=dev, dtype=dt) if (self.in_a is not None) else None
        if not self.static or not ours:
            R = torch.ones(B, self.S, device=dev, dtype=dt)       # resources ready (x in (d))
            q = torch.zeros(B, self.S, device=dev, dtype=dt)      # utilisation u
            y = torch.zeros(B, self.S, device=dev, dtype=dt) if not ours else None
        out = {}
        win = rec.get("windows")
        if win:
            wl, nw = win
            counts = torch.zeros(B, nw, N, device=dev, dtype=torch.int32)
        st_at = sorted(set(rec.get("states_at", [])))
        st_at_set = {t: i for i, t in enumerate(st_at)}
        if st_at:
            states = torch.zeros(B, len(st_at), N, device=dev, dtype=torch.float32)
        se = rec.get("states_every")
        if se:
            states_every = torch.zeros(B, se[1] - se[0], N, device=dev, dtype=torch.float32)
        nc = rec.get("neuron_counts")
        if nc:
            ncount = torch.zeros(B, N, device=dev, dtype=torch.int64)
        raster = torch.zeros(T * n, B, N, device=dev, dtype=torch.bool) if rec.get("raster") else None
        tr = {"u": [], "I1": [], "I2": [], "I3": [], "syn": []} if trace else None
        if open_loop is not None:
            open_loop = torch.as_tensor(open_loop, device=dev)
        if own_spikes is not None:
            own_spikes = torch.as_tensor(own_spikes, device=dev).bool()
            flips = torch.zeros((), device=dev, dtype=torch.int64)
        zero_in = torch.zeros(B, N, device=dev, dtype=dt)
        k = 0
        for t in range(T):
            step_spk = torch.zeros(B, N, device=dev, dtype=dt) if (win or nc) else None
            if t < T_in:
                in_push = x[:, t, :].to(dt) @ self.Win
            else:
                in_push = zero_in
            for j in range(n):
                blocked = ref > 0
                if ours:
                    # 1. synapses
                    if self.static:
                        pE = pI = None
                        for (dl, cls), W in self.groups.items():
                            if k - dl < 0:
                                continue
                            p = hist[(k - dl) % self.L] @ W
                            if cls == 0:
                                pE = p if pE is None else pE + p
                            else:
                                pI = p if pI is None else pI + p
                        pE = zero_in if pE is None else pE
                        pI = zero_in if pI is None else pI
                    else:
                        R = 1.0 - (1.0 - R) * self.eD
                        q = q * self.eF
                        a = self._arrivals(hist, k, B)
                        q = q + a * self.U * (1.0 - q)
                        push = a * self.wg * q * R
                        R = R - a * q * R
                        pE = push[:, :self.nE] @ self.M_E if self.nE else zero_in
                        pI = push[:, self.nE:] @ self.M_I if self.S > self.nE else zero_in
                    ip = in_push if j == 0 else zero_in
                    # 2. currents
                    if fine:
                        IE = self.aE * IE + (1 - self.aE) * pE
                        II = self.aI * II + (1 - self.aI) * pI
                        Iin = self.in_a * Iin + (1 - self.in_a) * ip
                        total = IE + II + Iin
                    else:
                        total = pE + pI + ip
                    # 3. integrate
                    u_new = self.beta * u + total + self.tap
                    if liq.clamp:
                        u_new = torch.where(blocked, u, u_new)
                    # 4. threshold and reset
                    if liq.snn_quirk:       # snnTorch: the spike test sees u_new - [u_prev > 1]
                        crossed = (u_new - (u > 1.0).to(dt)) > 1.0
                    elif self.adapt:
                        thr = 1.0 + self.a_c * ab
                        crossed = (u_new > thr) if liq.strict else (u_new >= thr)
                    else:
                        crossed = (u_new > 1.0) if liq.strict else (u_new >= 1.0)
                    spk = crossed & ~blocked
                    if own_spikes is not None:
                        flips += (spk != own_spikes[k]).sum()
                        spk = own_spikes[k]
                    if liq.reset_mode == "subtract":
                        u = u_new - spk.to(dt)
                    else:
                        u = torch.where(spk, torch.full_like(u_new, liq.reset_value), u_new)
                    if self.adapt:
                        ab = self.a_rho * ab + self.a_in * spk.to(dt)
                else:
                    # Brian2 schedule of (d)'s network (its sorted objects): synapse state update (exact), the
                    # summed current from the updated y, exact neuron update with that current held constant over
                    # the step and the input current decaying exactly; threshold; arrivals (on_pre); reset.
                    q = q * self.eF
                    R, y = R * self.eD + (1 - self.eD) + y * self.cxy, y * self.ey
                    # scatter-add per target, as Brian2's summed variable does (a runaway synapse of (d) as written
                    # then only affects its own target, instead of spreading NaN through a matrix product)
                    Jsyn = torch.zeros_like(zero_in).index_add_(1, self.post_idx, y * self.wg) if self.S else zero_in
                    u_new = self.beta * u + (1 - self.beta) * (self.J_b + Jsyn) + self.kappa * Iin
                    u_new = torch.where(blocked, u, u_new)
                    Iin = Iin * self.in_a
                    spk = (u_new > 1.0) & ~blocked
                    if own_spikes is not None:
                        flips += (spk != own_spikes[k]).sum()
                        spk = own_spikes[k]
                    if j == 0 and t < T_in:
                        Iin = Iin + in_push
                    hist[k % self.L] = (open_loop[k] if open_loop is not None else spk).to(dt)
                    a = self._arrivals(hist, k, B)
                    q = q + a * self.U * (1.0 - q)
                    y = y + a * q * R
                    R = R - a * q * R
                    u = torch.where(spk, torch.full_like(u_new, liq.reset_value), u_new)
                ref = torch.where(spk, self.gap - 1, torch.clamp(ref - 1, min=0))
                if ours:
                    hist[k % self.L] = (open_loop[k] if open_loop is not None else spk).to(dt)
                spf = spk.to(dt)
                z = z * self.z_decay + spf
                if step_spk is not None:
                    step_spk += spf
                if raster is not None:
                    raster[k] = spk
                if trace:
                    tr["u"].append(u.detach().cpu().numpy().copy())
                    if ours:
                        tr["I1"].append((IE if fine else pE).cpu().numpy().copy())
                        tr["I2"].append((II if fine else pI).cpu().numpy().copy())
                        tr["I3"].append((Iin if fine else ip).cpu().numpy().copy())
                    else:
                        tr["I1"].append(Jsyn.cpu().numpy().copy()); tr["I3"].append(Iin.cpu().numpy().copy())
                k += 1
            # end of input step t
            if win and t < wl * nw:
                counts[:, t // wl] += step_spk.to(torch.int32)
            if nc and nc[0] <= t < nc[1]:
                ncount += step_spk.to(torch.int64)
            if t in st_at_set:
                states[:, st_at_set[t]] = z.float()
            if se and se[0] <= t < se[1]:
                states_every[:, t - se[0]] = z.float()
            if trace and (not self.static or not ours):
                syn = {"R": R.cpu().numpy().copy(), "u": q.cpu().numpy().copy()}
                if not ours:
                    syn["y"] = y.cpu().numpy().copy()
                tr["syn"].append(syn)
        if win:
            out["windows"] = counts
        if st_at:
            out["states_at"] = states
            out["states_at_times"] = st_at
        if se:
            out["states_every"] = states_every
        if nc:
            out["neuron_counts"] = ncount
        if raster is not None:
            out["raster"] = raster
        if trace:
            out["trace"] = tr
        out["u_end"] = u
        if own_spikes is not None:
            out["flips"] = int(flips)
        return out


    # ------------------------------------------------------------------ fast path ("ours" schedule only)
    def _run_fast(self, x, u0, T, rec, graphs):
        """The same arithmetic as run()'s "ours" branch, in the same order, on fixed buffers: one input step is one
        call of `step` (or one replay of its CUDA graph). The spike-history ring holds m * n sub-steps (m = ceil(L / n))
        so the slots an input step reads depend only on t mod m; slots not yet written hold zeros, which is what the
        plain loop's skipped early arrivals add."""
        liq, dev, dt = self.liq, self.device, self.dtype
        B, T_in, Cin = x.shape
        N, n = self.N, liq.n_sub
        m = -(-self.L // n)
        Lr = m * n
        fine = liq.cur_a is not None
        st = {"u": self.start_state(u0), "ref": torch.zeros(B, N, device=dev, dtype=torch.int64),
              "hist": torch.zeros(Lr, B, N, device=dev, dtype=dt), "z": torch.zeros(B, N, device=dev, dtype=dt),
              "step": torch.zeros(B, N, device=dev, dtype=dt), "inp": torch.zeros(B, N, device=dev, dtype=dt)}
        if self.adapt:
            st["ab"] = torch.zeros(B, N, device=dev, dtype=dt)
        if fine:
            st["IE"] = torch.zeros(B, N, device=dev, dtype=dt)
            st["II"] = torch.zeros(B, N, device=dev, dtype=dt)
        if self.in_a is not None:
            st["Iin"] = torch.zeros(B, N, device=dev, dtype=dt)
        if not self.static:
            st["R"] = torch.ones(B, self.S, device=dev, dtype=dt)
            st["q"] = torch.zeros(B, self.S, device=dev, dtype=dt)
        zero_in = torch.zeros(B, N, device=dev, dtype=dt)

        if not self.static:
            # arrivals as gathers (capture-safe, no index_put): each delay group's presynaptic columns, then the
            # synapses back in their own order (synapses are sorted by type, so this is usually already in order)
            gpre = [(dl, self.pre_idx[idx]) for dl, idx in self.delay_groups]
            order = torch.cat([idx for _, idx in self.delay_groups])
            inv = None if torch.equal(order, torch.arange(self.S, device=dev)) else torch.argsort(order)

        def arrivals(hist, kk):
            a = torch.cat([hist[(kk - dl) % Lr].index_select(1, pre) for dl, pre in gpre], 1)
            return a if inv is None else a.index_select(1, inv)

        def step(phase):
            u, ref, hist, z = st["u"], st["ref"], st["hist"], st["z"]
            IE, II, Iin = st.get("IE"), st.get("II"), st.get("Iin")
            R, q = st.get("R"), st.get("q")
            ab = st.get("ab")
            step_spk = st["step"]
            step_spk.zero_()
            in_push = st["inp"]
            for j in range(n):
                kk = phase * n + j
                blocked = ref > 0
                if self.static:
                    pE = pI = None
                    for (dl, cls), W in self.groups.items():
                        p = hist[(kk - dl) % Lr] @ W
                        if cls == 0:
                            pE = p if pE is None else pE + p
                        else:
                            pI = p if pI is None else pI + p
                    pE = zero_in if pE is None else pE
                    pI = zero_in if pI is None else pI
                else:
                    R = 1.0 - (1.0 - R) * self.eD
                    q = q * self.eF
                    a = arrivals(hist, kk)
                    q = q + a * self.U * (1.0 - q)
                    push = a * self.wg * q * R
                    R = R - a * q * R
                    pE = push[:, :self.nE] @ self.M_E if self.nE else zero_in
                    pI = push[:, self.nE:] @ self.M_I if self.S > self.nE else zero_in
                ip = in_push if j == 0 else zero_in
                if fine:
                    IE = self.aE * IE + (1 - self.aE) * pE
                    II = self.aI * II + (1 - self.aI) * pI
                    Iin = self.in_a * Iin + (1 - self.in_a) * ip
                    total = IE + II + Iin
                else:
                    total = pE + pI + ip
                u_new = self.beta * u + total + self.tap
                if liq.clamp:
                    u_new = torch.where(blocked, u, u_new)
                if liq.snn_quirk:
                    crossed = (u_new - (u > 1.0).to(dt)) > 1.0
                elif self.adapt:
                    thr = 1.0 + self.a_c * ab
                    crossed = (u_new > thr) if liq.strict else (u_new >= thr)
                else:
                    crossed = (u_new > 1.0) if liq.strict else (u_new >= 1.0)
                spk = crossed & ~blocked
                if liq.reset_mode == "subtract":
                    u = u_new - spk.to(dt)
                else:
                    u = torch.where(spk, torch.full_like(u_new, liq.reset_value), u_new)
                if self.adapt:
                    ab = self.a_rho * ab + self.a_in * spk.to(dt)
                ref = torch.where(spk, self.gap - 1, torch.clamp(ref - 1, min=0))
                hist[kk % Lr] = spk.to(dt)
                spf = spk.to(dt)
                z = z * self.z_decay + spf
                step_spk += spf
            st["u"].copy_(u); st["ref"].copy_(ref); st["z"].copy_(z)
            if fine:
                st["IE"].copy_(IE); st["II"].copy_(II)
            if Iin is not None:
                st["Iin"].copy_(Iin)
            if R is not None:
                st["R"].copy_(R); st["q"].copy_(q)
            if ab is not None:
                st["ab"].copy_(ab)

        run_step = {ph: (lambda ph=ph: step(ph)) for ph in range(m)}
        if graphs:
            backup = {k: v.clone() for k, v in st.items()}
            side = torch.cuda.Stream(device=dev)
            side.wait_stream(torch.cuda.current_stream(dev))
            with torch.cuda.stream(side):
                for ph in range(m):
                    step(ph)                                   # warm-up (cuBLAS handles, allocator)
            torch.cuda.current_stream(dev).wait_stream(side)
            pool = None
            for ph in range(m):
                g = torch.cuda.CUDAGraph()
                with torch.cuda.graph(g, pool=pool):
                    step(ph)
                pool = g.pool()
                run_step[ph] = g.replay
            for k, v in st.items():                            # undo the warm-up's changes to the state
                v.copy_(backup[k])
            del backup

        out = {}
        win = rec.get("windows")
        if win:
            wl, nw = win
            counts = torch.zeros(B, nw, N, device=dev, dtype=torch.int32)
        st_at = sorted(set(rec.get("states_at", [])))
        st_at_set = {t: i for i, t in enumerate(st_at)}
        if st_at:
            states = torch.zeros(B, len(st_at), N, device=dev, dtype=torch.float32)
        se = rec.get("states_every")
        if se:
            states_every = torch.zeros(B, se[1] - se[0], N, device=dev, dtype=torch.float32)
        nc = rec.get("neuron_counts")
        if nc:
            ncount = torch.zeros(B, N, device=dev, dtype=torch.int64)
        raster = torch.zeros(T * n, B, N, device=dev, dtype=torch.bool) if rec.get("raster") else None
        if raster is not None:
            raise ValueError("the fast path does not record rasters: use fast=False")
        for t in range(T):
            if t < T_in:
                st["inp"].copy_(x[:, t, :].to(dt) @ self.Win)
            else:
                st["inp"].zero_()
            run_step[t % m]()
            step_spk, z = st["step"], st["z"]
            if win and t < wl * nw:
                counts[:, t // wl] += step_spk.to(torch.int32)
            if nc and nc[0] <= t < nc[1]:
                ncount += step_spk.to(torch.int64)
            if t in st_at_set:
                states[:, st_at_set[t]] = z.float()
            if se and se[0] <= t < se[1]:
                states_every[:, t - se[0]] = z.float()
        if win:
            out["windows"] = counts
        if st_at:
            out["states_at"] = states
            out["states_at_times"] = st_at
        if se:
            out["states_every"] = states_every
        if nc:
            out["neuron_counts"] = ncount
        out["u_end"] = st["u"].clone()
        return out


# ---------------------------------------------------------------- starting states (CPU, from (seed, beat index))
def start_voltages(liq, seed, beat_ids, stream, random_start=None):
    """Starting voltages for each beat, drawn on the CPU from (seed, stream, beat index) so they do not depend
    on batching or the card. random_start=None: the liquid's own rule (0 in X; uniform 0.9..1 with the tap).
    random_start=True: uniform between the resting level and the threshold (level 3's two starting states)."""
    lo, hi = (liq.rest, 1.0) if random_start else (liq.start_lo, liq.start_hi)
    out = np.empty((len(beat_ids), liq.N))
    for i, b in enumerate(beat_ids):
        if hi > lo:
            out[i] = np.random.default_rng([int(seed), int(stream), int(b)]).uniform(lo, hi, liq.N)
        else:
            out[i] = lo
    return out
'''
LSM2_FILES['lsm2/stats.py'] = r'''"""Statistics and decision rules (plan, Statistics and decision rules): the paired Nadeau-Bengio corrected
t-test over the 12 CV splits (variance factor 1/12 + 0.336), Holm within a family, the DS2 difference paired by
seed, and the verdicts meaningful / comparable / inconclusive. Inputs are tables of runs with columns spec_split,
spec_seed, macro_f1 (and DS2 metrics)."""
import math
import numpy as np
from scipy import stats as sstats

NB_RATIO = 0.336          # plan: test/train ratio of the 3x4 patient-grouped CV (the replication used 0.25/0.75)


def per_split_seed(d):
    """Macro-F1 by (split, seed)."""
    return d.groupby(["spec_split", "spec_seed"], dropna=False).macro_f1.mean()

def compare(a, b, require_seed_sign=False):
    """A vs B (Macro-F1): seeds averaged per CV split; paired Nadeau-Bengio corrected t-test over the 12 splits
    with its 95% interval; the DS2 difference paired by seed."""
    if a is None or b is None or a.empty or b.empty:
        return None
    A, B = per_split_seed(a), per_split_seed(b)
    if a["spec_seed"].isna().all() or b["spec_seed"].isna().all():      # a seed-free control pairs with every seed
        A = A.groupby(level=0).mean(); B = B.groupby(level=0).mean()
        Asp, Bsp = A, B
        ds2_seed = None
    else:
        Asp, Bsp = A.groupby(level=0).mean(), B.groupby(level=0).mean()
        ds2_seed = (A.xs("final", level=0) - B.xs("final", level=0)).dropna() if "final" in A.index.get_level_values(0) and "final" in B.index.get_level_values(0) else None
    cv = [s for s in Asp.index if str(s).startswith("cv-") and s in Bsp.index]
    if len(cv) < 3:
        return None
    d = (Asp[cv] - Bsp[cv]).values
    k = len(d)
    var = np.var(d, ddof=1) * (1 / k + NB_RATIO)
    se = math.sqrt(var)
    t = d.mean() / se if se > 0 else np.inf
    p = 2 * sstats.t.sf(abs(t), k - 1)
    q = sstats.t.ppf(0.975, k - 1)
    ds2 = float(Asp.get("final", np.nan) - Bsp.get("final", np.nan)) if ("final" in Asp.index and "final" in Bsp.index) else np.nan
    out = {"n_splits": k, "cv_diff": float(d.mean()), "ci_low": float(d.mean() - q * se), "ci_high": float(d.mean() + q * se),
           "t": float(t), "p": float(p), "ds2_diff": ds2,
           "ds2_diff_by_seed": None if ds2_seed is None else {str(s): float(v) for s, v in ds2_seed.items()}}
    # DS2 without record 232, and per-class SVEB / VEB recall and precision (plan: every comparison)
    fa, fb = a[a.spec_split == "final"], b[b.spec_split == "final"]
    for m in ("macro_f1_no232", "recall_SVEB", "precision_SVEB", "recall_VEB", "precision_VEB"):
        if m in fa and m in fb and len(fa) and len(fb):
            out[f"ds2_diff_{m}"] = float(fa[m].mean() - fb[m].mean())
    if require_seed_sign and not a["spec_seed"].isna().all():
        per_seed = []
        for sd in sorted(set(a.spec_seed.dropna()) & set(b.spec_seed.dropna())):
            xa = a[(a.spec_seed == sd) & a.spec_split.astype(str).str.startswith("cv-")].groupby("spec_split").macro_f1.mean()
            xb = b[(b.spec_seed == sd) & b.spec_split.astype(str).str.startswith("cv-")].groupby("spec_split").macro_f1.mean()
            j = xa.index.intersection(xb.index)
            per_seed.append(float((xa[j] - xb[j]).mean()))
        out["cv_diff_by_seed"] = per_seed
        out["same_sign_all_seeds"] = bool(per_seed and (all(x > 0 for x in per_seed) or all(x < 0 for x in per_seed)))
    return out

def compare_diff(a_on, a_off, b_on, b_off):
    """F7: (on - off, full) - (on - off, ablated), per split (seeds averaged), corrected t-test."""
    parts = [per_split_seed(x).groupby(level=0).mean() if x is not None and not x.empty else None for x in (a_on, a_off, b_on, b_off)]
    if any(p is None for p in parts):
        return None
    D = (parts[0] - parts[1]) - (parts[2] - parts[3])
    cv = [s for s in D.index if str(s).startswith("cv-") and np.isfinite(D[s])]
    if len(cv) < 3:
        return None
    d = D[cv].values; k = len(d)
    se = math.sqrt(np.var(d, ddof=1) * (1 / k + NB_RATIO)); t = d.mean() / se if se > 0 else np.inf
    q = sstats.t.ppf(0.975, k - 1)
    return {"n_splits": k, "cv_diff": float(d.mean()), "ci_low": float(d.mean() - q * se), "ci_high": float(d.mean() + q * se),
            "t": float(t), "p": float(2 * sstats.t.sf(abs(t), k - 1)), "ds2_diff": float(D.get("final", np.nan)),
            "plain_full_minus_ablated_on": float((parts[0] - parts[2])[cv].mean())}

def holm(rows):
    ps = [r["p"] if r else np.nan for r in rows]
    idx = [i for i, p in enumerate(ps) if np.isfinite(p)]
    order = sorted(idx, key=lambda i: ps[i]); m = len(order); run = 0.0
    for rank, i in enumerate(order):
        run = max(run, min(1.0, (m - rank) * ps[i]))
        rows[i]["p_holm"] = run
    return rows

def verdict(r, need_seed_sign=False):
    if r is None:
        return "missing"
    meaningful = abs(r["cv_diff"]) >= 0.01 and r.get("p_holm", 1) < 0.05 and np.sign(r["ds2_diff"]) == np.sign(r["cv_diff"])
    if need_seed_sign:
        meaningful = meaningful and r.get("same_sign_all_seeds", False)
    if meaningful:
        return "meaningful"
    if r["ci_low"] >= -0.02 and r["ci_high"] <= 0.02 and abs(r["ds2_diff"]) < 0.02:
        return "comparable"
    return "inconclusive"

def family(name, items, need_seed_sign=False):
    rows = [it[1] for it in items]
    rows = holm([dict(r) if r else None for r in rows])
    out = []
    for (label, _), r in zip(items, rows):
        out.append({"family": name, "comparison": label, **({} if r is None else {k: v for k, v in r.items() if not isinstance(v, (dict, list))}),
                    "verdict": verdict(r, need_seed_sign)})
    return out

'''

import shutil
shutil.rmtree(os.path.join(SCRATCH_DIR, "lsm2"), ignore_errors=True)
for _rel, _src in LSM2_FILES.items():
    _p = os.path.join(SCRATCH_DIR, _rel)
    os.makedirs(os.path.dirname(_p), exist_ok=True)
    with open(_p, "w") as _f:
        _f.write(_src)
open(os.path.join(SCRATCH_DIR, "lsm2", "__init__.py"), "w").close()
if SCRATCH_DIR not in sys.path:
    sys.path.insert(0, SCRATCH_DIR)
for _m in [m for m in sys.modules if m == "lsm2" or m.startswith("lsm2.")]:
    del sys.modules[_m]
import lsm2
print(f"lsm2: {len(LSM2_FILES)} files at {os.path.join(SCRATCH_DIR, 'lsm2')}")


## 4. Utilities: logging, metrics, the authors' readout

In [ ]:
# Utilities: output folders, logging, cache.txt, resume, time/disk guards, metrics, the authors' readout.
CODE_VERSION = "v1"   # part of every run's hash; bump only if experiment code changes meaning
for sub in ["runs", "reservoirs", "states", "tables", "figures", "logs", "checks"]:
    os.makedirs(os.path.join(WORK_DIR, sub), exist_ok=True)

def elapsed_h():
    return (time.time() - SESSION_START) / 3600

LOCK = threading.RLock()     # cache.txt, the log, speeds and disk counters are shared with the fit workers

def log(*parts):
    line = f"[{elapsed_h():6.2f} h] " + " ".join(str(p) for p in parts)
    with LOCK:
        print(line, flush=True)               # the tee writes it to LOG_PATH

def _json_default(o):
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating,)): return float(o)
    if isinstance(o, np.ndarray): return o.tolist()
    if isinstance(o, (np.bool_,)): return bool(o)
    if isinstance(o, torch.Tensor): return o.detach().cpu().tolist()
    return str(o)

def atomic_write_text(path, text):
    tmp = path + ".tmp"
    with open(tmp, "w") as f:
        f.write(text)
    os.replace(tmp, path)

def jdump(obj, path):
    atomic_write_text(path, json.dumps(obj, indent=2, default=_json_default))

def stable_hash(obj, n=8):
    return hashlib.sha1(json.dumps(obj, sort_keys=True, default=_json_default).encode()).hexdigest()[:n]

# ---------------- cache.txt ----------------
CACHE_PATH = os.path.join(WORK_DIR, "cache.txt")
MACHINE_ID = f"{MACHINE}|{ENV.get('gpu', DEVICE.type)}|{N_CPU} cores"      # identifies the machine in merged outputs

def out_tag(share, machine_id):
    """Folder-safe name of an output: its share and machine."""
    return re.sub(r"[^A-Za-z0-9.]+", "_", f"{share if share not in (None, 'None') else 'all'}__{machine_id or 'unknown'}").strip("_")

class RunCache:
    """cache.txt = the record of what has run. Rewritten (atomically) after every run.
    Sections: [DONE] run_id, finished_utc, seconds, macro_f1, accuracy, group
              [FAILED] run_id, when_utc, error
              [PENDING] run_id, group"""
    def __init__(self):
        self.done, self.failed, self.pending = {}, {}, []

    def merge_file(self, path):
        if not os.path.exists(path):
            return
        section = None
        for line in open(path):
            line = line.rstrip("\n")
            if line.startswith("[") and line.endswith("]"):
                section = line[1:-1]
                continue
            if not line or line.startswith("#"):
                continue
            parts = line.split("\t")
            if section == "DONE":
                self.done[parts[0]] = parts[1:]
                self.failed.pop(parts[0], None)
            elif section == "FAILED" and parts[0] not in self.done:
                self.failed[parts[0]] = parts[1:]

    def is_done(self, rid):
        return rid in self.done

    def mark_done(self, rid, seconds, metrics, group):
        with LOCK:
            self._mark_done(rid, seconds, metrics, group)

    def _mark_done(self, rid, seconds, metrics, group):
        mf1 = f"{metrics['macro_f1']:.4f}" if metrics and "macro_f1" in metrics else "-"
        acc = f"{metrics['accuracy']:.4f}" if metrics and "accuracy" in metrics else "-"
        self.done[rid] = [datetime.datetime.utcnow().isoformat(timespec="seconds"), f"{seconds:.1f}", mf1, acc, group]
        self.failed.pop(rid, None)
        self.write()

    def mark_failed(self, rid, err):
        with LOCK:
            self._mark_failed(rid, err)

    def _mark_failed(self, rid, err):
        self.failed[rid] = [datetime.datetime.utcnow().isoformat(timespec="seconds"), err.replace("\t", " ").replace("\n", " | ")[:500]]
        self.write()

    def write(self):
        with LOCK:
            self._write()

    def _write(self):
        lines = ["# LSM-ECG replication: run cache. Rewritten after every run; read by the next session to skip finished work.",
                 f"# updated_utc: {datetime.datetime.utcnow().isoformat(timespec='seconds')}   session_elapsed_h: {elapsed_h():.2f}",
                 f"# totals: DONE {len(self.done)} | FAILED {len(self.failed)} | PENDING {len(self.pending)}",
                 f"# smoke_test: {SMOKE_TEST}",
                 f"# share: {SHARE}",
                 f"# machine_id: {MACHINE_ID}",
                 f"# code_version: {CODE_VERSION}",
                 f"# notebook_build: {NOTEBOOK_BUILD}",
                 "# DONE columns: run_id, finished_utc, seconds, macro_f1, accuracy, group", "[DONE]"]
        lines += ["\t".join([k] + v) for k, v in self.done.items()]
        lines += ["", "# FAILED columns: run_id, when_utc, error", "[FAILED]"]
        lines += ["\t".join([k] + v) for k, v in self.failed.items()]
        lines += ["", "# PENDING columns: run_id, group", "[PENDING]"]
        lines += [f"{rid}\t{g}" for rid, g in self.pending]
        atomic_write_text(CACHE_PATH, "\n".join(lines) + "\n")

CACHE = RunCache()

# ---------------- resume (previous session) and merge mode (two machines' outputs) ----------------
RESUME_SUBDIRS = ["runs", "reservoirs", "states", "logs", "checks"]   # outputs worth carrying; tables/figures are rebuilt

def _cache_is_smoke(p):
    return any(l.strip() == "# smoke_test: True" for l in itertools.islice(open(p), 12))

def cache_header(p):
    h = {}
    for l in itertools.islice(open(p), 12):
        m = re.match(r"^# (smoke_test|share|machine_id|code_version|notebook_build): (.*)$", l.rstrip("\n"))
        if m:
            h[m.group(1)] = m.group(2).strip()
    return h

def copy_forward(src_root):
    n = 0
    for sub in RESUME_SUBDIRS:
        for root, _, files in os.walk(os.path.join(src_root, sub)):
            rel = os.path.relpath(root, src_root)
            for fn in files:
                dst = os.path.normpath(os.path.join(WORK_DIR, rel, fn))
                if not os.path.exists(dst):
                    os.makedirs(os.path.dirname(dst), exist_ok=True)
                    shutil.copy2(os.path.join(root, fn), dst)
                    n += 1
    return n

if MERGE_MODE:
    for md in MERGE_DIRS:
        if os.path.abspath(md) == os.path.abspath(WORK_DIR):
            raise ValueError("merge mode: WORK_DIR must be a new folder, not one of MERGE_DIRS")
        if not os.path.exists(os.path.join(md, "cache.txt")):
            raise FileNotFoundError(f"merge mode: no cache.txt in {md} (give the output folders themselves)")
    _hdrs = [cache_header(os.path.join(md, "cache.txt")) for md in MERGE_DIRS]
    if len({h.get("notebook_build") for h in _hdrs}) > 1:
        print("WARNING: the folders were made by different notebook builds: "
              + ", ".join(f"{md}: {h.get('notebook_build', '?')}" for md, h in zip(MERGE_DIRS, _hdrs)))

if RESUME_DIRS == "auto":
    RESUME_DIRS = sorted({os.path.dirname(p) for p in glob.glob(os.path.join(AUTO_RESUME_ROOT, "**", "cache.txt"), recursive=True)})
for rd in (MERGE_DIRS if MERGE_MODE else RESUME_DIRS):
    cp = os.path.join(rd, "cache.txt")
    if os.path.abspath(rd) == os.path.abspath(WORK_DIR):
        continue
    if not os.path.exists(cp):
        print(f"resume: no cache.txt in {rd} -> fresh start")
        continue
    if _cache_is_smoke(cp) != SMOKE_TEST:
        msg = f"{rd}: its smoke_test flag differs from this session's SMOKE_TEST={SMOKE_TEST}"
        if MERGE_MODE:
            raise ValueError("merge mode: " + msg)
        print(f"resume: skipping {msg}")
        continue
    before = len(CACHE.done)
    _o = RunCache(); _o.merge_file(cp); n_there = len(_o.done)
    CACHE.merge_file(cp)
    n = copy_forward(rd)
    n_dirs = len(os.listdir(os.path.join(rd, "runs"))) if os.path.isdir(os.path.join(rd, "runs")) else 0
    h = cache_header(cp)
    print(f"{'merged' if MERGE_MODE else 'resumed'} from {rd} (share {h.get('share', '?')}, {h.get('machine_id', '?')}): "
          f"{n_there} finished runs there, {len(CACHE.done) - before} of them new here, {n} files copied")
    if n_dirs < len(CACHE.done) - before - 2:
        print(f"WARNING: its cache.txt lists {len(CACHE.done) - before} finished runs but only {n_dirs} run folders are there. "
              "They will be skipped, but their results will be missing from tables and figures. "
              "Bring the whole output folder, not only cache.txt.")
if os.path.exists(CACHE_PATH) and _cache_is_smoke(CACHE_PATH) != SMOKE_TEST:
    raise RuntimeError(f"{WORK_DIR} holds {'smoke-test' if _cache_is_smoke(CACHE_PATH) else 'real'} results but SMOKE_TEST = "
                       f"{SMOKE_TEST}. Use a separate WORK_DIR for the smoke test so the two never mix.")
CACHE.merge_file(CACHE_PATH)       # this machine's earlier sessions (same folder, reused in place)
print(f"cache: {len(CACHE.done)} done, {len(CACHE.failed)} failed")

# ---------------- time and disk guards ----------------
def seconds_left():
    return SESSION_BUDGET_HOURS * 3600 - (time.time() - SESSION_START) - ANALYSIS_RESERVE_MIN * 60

_DISK = {"bytes": None, "since": 0}
def work_dir_gb(force=False):
    if force or _DISK["bytes"] is None or _DISK["since"] >= 50:
        total = 0
        for root, _, files in os.walk(WORK_DIR):
            for fn in files:
                try:
                    total += os.path.getsize(os.path.join(root, fn))
                except OSError:
                    pass
        _DISK.update(bytes=total, since=0)
    return _DISK["bytes"] / 1e9

def disk_add(nbytes):
    with LOCK:
        if _DISK["bytes"] is not None:
            _DISK["bytes"] += nbytes
            _DISK["since"] += 1

# Throughput model, measured as the session runs (and carried over between sessions in checks/speeds.json).
# Units: beats/s for simulation and encoding; seconds per (training beats x features / 1e9) for a classifier fit;
# seconds per epoch per 1,000 training beats for the CNN/GRU.
_GPU = DEVICE.type == "cuda"
# GPU defaults: roughly what a Kaggle T4 measured (used only until this machine has measured its own)
SPEED = {"sim_beats_per_s": {e: (2000.0 if _GPU else 300.0) for e in ("multi", "rate", "latency")},
         "encode_beats_per_s": {e: (20000.0 if _GPU else 1500.0) for e in ("multi", "rate", "latency")},
         "fit_s_per_Gcell": {"1": 400.0, "2": 220.0, "3": 160.0, "pipe": 400.0}, "fit_threads": float(READOUT_JOBS),
         "nn_s_per_epoch_kbeat": {"cnn": 0.04 if _GPU else 3.0, "gru": 0.025 if _GPU else 8.0},
         "nn_unit_s_per_epoch_kbeat": {"cnn": 0.03 if _GPU else 3.0, "gru": 0.04 if _GPU else 8.0},
         "lsm_job_overhead_s": 3.0, "huang_s_per_run": 1200.0}
SPEED_SAMPLES = {}                # recent measurements of the robust (median) entries
# fit_s_per_Gcell: "1"/"2"/"3" = one fit using that many cores (older runner); "pipe" = one fit per core while all
# the pipeline's workers are busy (includes their competition for memory bandwidth).
# nn_unit_*: seconds per epoch per 1,000 beats for a tuned architecture, divided by its relative cost (see nn_cost).
SPEED_MEASURED = set()            # which entries were measured (vs default guesses)
SPEEDS_PATH = os.path.join(WORK_DIR, "checks", "speeds.json")
if os.path.exists(SPEEDS_PATH):
    try:
        _sp = json.load(open(SPEEDS_PATH))
        if _sp.get("device") == DEVICE.type and _sp.get("gpu") == ENV.get("gpu") and _sp.get("cpu_count") == N_CPU:
            for k, v in _sp["speed"].items():
                if isinstance(SPEED.get(k), dict) and isinstance(v, dict):
                    SPEED[k].update(v)
                elif k in SPEED and not isinstance(SPEED[k], dict) and not isinstance(v, dict):
                    SPEED[k] = v                                               # entries whose format changed are skipped
            SPEED_MEASURED.update(m for m in _sp["measured"] if not (m == "fit_s_per_Gcell"))
            SPEED_SAMPLES.update(_sp.get("samples", {}))
            print("speeds carried over from an earlier session:", sorted(SPEED_MEASURED))
    except Exception as ex:
        print("could not read speeds.json:", ex)

def update_speed(key, value, sub=None, alpha=0.3, robust=False):
    with LOCK:
        if robust:            # median of the last 40 measurements: one odd fit does not move the projection
            tag = f"{key}:{sub}" if sub else key
            xs = SPEED_SAMPLES.setdefault(tag, []); xs.append(float(value)); del xs[:-40]
            value, alpha = float(np.median(xs)), 1.0
        _update_speed(key, value, sub, alpha)

def _update_speed(key, value, sub, alpha):
    tag = f"{key}:{sub}" if sub else key
    if sub is None:
        SPEED[key] = value if tag not in SPEED_MEASURED else alpha * value + (1 - alpha) * SPEED[key]
    else:
        SPEED[key][sub] = value if tag not in SPEED_MEASURED else alpha * value + (1 - alpha) * SPEED[key][sub]
    SPEED_MEASURED.add(tag)

def save_speeds():
    with LOCK:
        jdump({"device": DEVICE.type, "gpu": ENV.get("gpu"), "cpu_count": N_CPU, "speed": SPEED, "samples": SPEED_SAMPLES, "measured": sorted(SPEED_MEASURED)}, SPEEDS_PATH)

# ---------------- metrics ----------------
CLASS_NAMES = ["N", "SVEB", "VEB"]

def compute_metrics(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2])
    tp = np.diag(cm).astype(float)
    support, predicted, total = cm.sum(1).astype(float), cm.sum(0).astype(float), cm.sum()
    rec = np.divide(tp, support, out=np.zeros(3), where=support > 0)
    prec = np.divide(tp, predicted, out=np.zeros(3), where=predicted > 0)
    f1 = np.divide(2 * prec * rec, prec + rec, out=np.zeros(3), where=(prec + rec) > 0)
    fp = predicted - tp
    tn = total - support - fp
    spec = np.divide(tn, tn + fp, out=np.zeros(3), where=(tn + fp) > 0)
    m = {"accuracy": float(tp.sum() / total), "macro_f1": float(f1.mean()), "macro_recall": float(rec.mean()),
         "macro_precision": float(prec.mean()), "n": int(total), "confusion_matrix": cm.tolist()}
    for i, c in enumerate(CLASS_NAMES):
        m.update({f"recall_{c}": float(rec[i]), f"precision_{c}": float(prec[i]), f"f1_{c}": float(f1[i]),
                  f"specificity_{c}": float(spec[i]), f"fpr_{c}": float(1 - spec[i]), f"support_{c}": int(support[i])})
    return m

# ---------------- the authors' readout (exact settings) ----------------
def soft_class_weight(y, p=0.6):
    classes = np.unique(y)
    base = compute_class_weight(class_weight="balanced", classes=classes, y=y).astype(np.float64)
    return dict(zip(classes.astype(int), (base ** p).astype(float)))

CURVE_EVAL_EPOCHS = {1, 3, 5, 10, 15}      # epochs at which the validation curve is measured (CV runs only)

class Cancelled(Exception):
    """Raised inside a fit when the session's time is up; the run stays pending for the next session."""

def prepare_train(Xtr_fn, ntr, prescale):
    """StandardScaler fitted with batched partial_fit (authors' way); optionally the whole scaled training matrix."""
    bs = 4096
    scaler = StandardScaler(with_mean=True, with_std=True)
    for i in range(0, ntr, bs):
        scaler.partial_fit(Xtr_fn(np.arange(i, min(i + bs, ntr))))
    XS = None
    if prescale:
        XS = np.concatenate([scaler.transform(Xtr_fn(np.arange(i, min(i + bs, ntr)))).astype(np.float32) for i in range(0, ntr, bs)])
    return scaler, XS

def fit_readout(Xtr_fn, ntr, ytr, Xev_fn, nev, loss, rng_seed, yev_for_curve=None, curve_sub=4000, prescale=False, n_jobs=None,
                prepared=None, cancel=None, epochs=None):
    """Authors' readout: StandardScaler (batched partial_fit) + SGDClassifier(hinge|log_loss, l2, alpha=1e-4,
    optimal LR, average=True, random_state=2026, soft class weights ^0.6), 15 epochs, batch 4096,
    shuffling with np.random.default_rng(rng_seed). Batches are scaled on the fly as in the authors' code, or, when
    prescale=True and RAM allows, the whole matrix is scaled once (row-wise identical values, faster).
    prepared=(scaler, XS) reuses a scaler / scaled matrix shared by several fits on the same training rows.
    Xtr_fn(positions) / Xev_fn(positions) return float32 feature rows."""
    bs = 4096
    scaler, XS = prepared if prepared is not None else prepare_train(Xtr_fn, ntr, prescale)
    tf = lambda X: scaler.transform(X).astype(np.float32)
    get_tr = (lambda p: XS[p]) if XS is not None else (lambda p: tf(Xtr_fn(p)))
    clf = SGDClassifier(loss=loss, penalty="l2", alpha=1e-4, fit_intercept=True, learning_rate="optimal",
                        max_iter=1, tol=None, class_weight=soft_class_weight(ytr), average=True, random_state=2026,
                        n_jobs=n_jobs)   # n_jobs only parallelises the 3 one-vs-rest problems: results are bit-identical
    classes = np.unique(ytr)
    rng = np.random.default_rng(rng_seed)
    sub = np.sort(np.random.default_rng(0).choice(ntr, min(curve_sub, ntr), replace=False))
    Xsub = get_tr(sub)
    def predict_eval():
        preds, scores = [], []
        for i in range(0, nev, bs):
            Xe = tf(Xev_fn(np.arange(i, min(i + bs, nev))))
            preds.append(clf.predict(Xe)); scores.append(clf.decision_function(Xe).astype(np.float32))
        return np.concatenate(preds), np.concatenate(scores)
    curve = []
    for ep in range(epochs or READOUT_EPOCHS):          # epochs: only the core search uses fewer
        idx = rng.permutation(ntr)
        for i in range(0, ntr, bs):
            if cancel is not None and cancel.is_set():
                raise Cancelled()
            sel = idx[i:i + bs]
            Xb = get_tr(sel)
            if ep == 0 and i == 0:
                clf.partial_fit(Xb, ytr[sel], classes=classes)
            else:
                clf.partial_fit(Xb, ytr[sel])
        row = {"epoch": ep + 1, "train_sub_macro_f1": compute_metrics(ytr[sub], clf.predict(Xsub))["macro_f1"]}
        if loss == "hinge":
            dec = clf.decision_function(Xsub)
            onehot = -np.ones_like(dec); onehot[np.arange(len(sub)), ytr[sub]] = 1
            row["train_sub_hinge_loss"] = float(np.maximum(0, 1 - onehot * dec).mean())
        if yev_for_curve is not None and (ep + 1) in CURVE_EVAL_EPOCHS:
            row["eval_macro_f1"] = compute_metrics(yev_for_curve, predict_eval()[0])["macro_f1"]
        curve.append(row)
    y_pred, scores = predict_eval()
    return scaler, clf, y_pred, scores, curve

# ---------------- saving a run ----------------
def save_run(rid, spec, group, metrics, eval_idx, eval_pid, y_true, y_pred, scores, model_obj,
             curve, liquid_stats, timing, extra=None):
    d = os.path.join(WORK_DIR, "runs", rid)
    os.makedirs(d, exist_ok=True)
    jdump({"run_id": rid, "group": group, "spec": spec, "env": ENV, "code_version": CODE_VERSION,
           "session": SESSION_TAG, **(extra or {})}, os.path.join(d, "config.json"))
    jdump(metrics, os.path.join(d, "metrics.json"))
    np.savez_compressed(os.path.join(d, "predictions.npz"), eval_index=np.asarray(eval_idx, np.int32),
                        patient_id=np.asarray(eval_pid).astype("U4"), y_true=np.asarray(y_true, np.int8),
                        y_pred=np.asarray(y_pred, np.int8),
                        scores=np.asarray(scores, np.float32) if scores is not None else np.zeros((0, 3), np.float32))
    if model_obj is not None:
        if isinstance(model_obj, dict) and model_obj.get("_torch_state"):
            torch.save(model_obj, os.path.join(d, "model.pt"))
        else:
            joblib.dump(model_obj, os.path.join(d, "readout.joblib"), compress=3)
    if curve:
        pd.DataFrame(curve).to_csv(os.path.join(d, "train_curve.csv"), index=False)
    if liquid_stats:
        jdump(liquid_stats, os.path.join(d, "liquid_stats.json"))
    jdump(timing, os.path.join(d, "timing.json"))
    with open(os.path.join(d, "log.txt"), "w") as f:
        f.write(f"{rid}\nmacro_f1={metrics.get('macro_f1')}\naccuracy={metrics.get('accuracy')}\n"
                f"confusion_matrix={metrics.get('confusion_matrix')}\ntiming={timing}\n")
    disk_add(sum(os.path.getsize(os.path.join(d, f)) for f in os.listdir(d)))


## 5. Data: MIT-BIH → DS1 / DS2 and the authors' folds

In [ ]:
# Data: locate + checksum MIT-BIH, build DS1/DS2 exactly as the authors do (both filters),
# track each beat's position for RR-interval features, verify against the authors' released files.
def locate_input():
    cands = [INPUT_DIR] + sorted({os.path.dirname(p) for p in glob.glob("/kaggle/input/**/100.hea", recursive=True)})
    for c in cands:
        if os.path.exists(os.path.join(c, "100.hea")) and os.path.exists(os.path.join(c, "234.atr")):
            return c
    raise FileNotFoundError(f"MIT-BIH not found at {INPUT_DIR} or anywhere under /kaggle/input")

DATA_DIR = locate_input()
print("MIT-BIH at:", DATA_DIR)

CHECKS = {}
_recs = pd.read_csv(os.path.join(REPO_DIR, "splits/ds1_ds2_records.csv")).set_index("set")["records"]
DS_RECORDS = {"DS1": _recs["DS1"].split(","), "DS2": _recs["DS2"].split(",")}
DS_RECORDS_ALL = DS_RECORDS["DS1"] + DS_RECORDS["DS2"]
sums_path = os.path.join(DATA_DIR, "SHA256SUMS.txt")
if os.path.exists(sums_path):
    bad, missing, n = [], [], 0
    for line in open(sums_path):
        h, fn = line.strip().split(maxsplit=1)
        fp = os.path.join(DATA_DIR, fn)
        if os.path.isfile(fp):
            n += 1
            if hashlib.sha256(open(fp, "rb").read()).hexdigest() != h:
                bad.append(fn)
        else:
            missing.append(fn)
    used = {f"{r}.{x}" for r in DS_RECORDS_ALL for x in ("dat", "hea", "atr")}
    CHECKS["checksums"] = {"files_listed": n + len(missing), "files_checked": n, "mismatches": bad, "missing": missing,
                           "missing_files_used_by_pipeline": sorted(set(missing) & used)}
    print(f"checksums: {n} files checked, {len(bad)} mismatches, {len(missing)} missing"
          + (f" ({', '.join(missing[:12])}{' ...' if len(missing) > 12 else ''})" if missing else ""))
    if set(missing) & used:
        raise FileNotFoundError(f"record files needed by the pipeline are missing: {sorted(set(missing) & used)}")
else:
    CHECKS["checksums"] = {"files_checked": 0, "note": "SHA256SUMS.txt not present"}


NORMAL_S, SVEB_S, VEB_S = ['N', 'L', 'R', 'e', 'j'], ['A', 'a', 'J', 'S'], ['V', 'E']
BEAT_SYMBOLS = set("NLRBAaJSVrFejnE/fQ?")      # all beat annotations (for RR intervals)
RR_NAMES = ["pre", "post", "local10", "rec_median", "pre_over_local", "post_over_local", "pre_over_rec", "pre_over_post"]

def rr_features(all_beats, pos, fs):
    """RR features (seconds / ratios) for beats at sample positions `pos`, from every beat annotation."""
    all_beats = np.sort(np.asarray(all_beats))
    rr = np.diff(all_beats) / fs
    rec_med = float(np.median(rr))
    i = np.searchsorted(all_beats, pos)
    pre = np.where(i > 0, rr[np.clip(i - 1, 0, len(rr) - 1)], rec_med)
    post = np.where(i < len(all_beats) - 1, rr[np.clip(i, 0, len(rr) - 1)], rec_med)
    csum = np.concatenate([[0], np.cumsum(rr)])
    lo = np.clip(i - 10, 0, None)
    cnt = np.maximum(i - lo, 1)
    local = np.where(i > 0, (csum[np.clip(i, 0, len(rr))] - csum[lo]) / cnt, rec_med)
    f = np.stack([pre, post, local, np.full_like(pre, rec_med), pre / local, post / local, pre / rec_med, pre / post], 1)
    return f.astype(np.float32)

def rr_mean_ratio(all_beats, pos, fs):
    """pre-RR / mean RR of the whole recording (Huang et al. 2014); the same pre-RR as the "pre" column."""
    rr = np.diff(np.sort(np.asarray(all_beats))) / fs
    return (rr_features(all_beats, pos, fs)[:, 0] / rr.mean()).astype(np.float32)

def huang_windows(sig, pos, half=100):
    """Huang et al.: 200 raw (unfiltered) samples, R-100 .. R+99 at 360 Hz, minus their mean, divided by their SD."""
    W = np.stack([sig[p - half:p + half] for p in pos]).astype(np.float64)
    return ((W - W.mean(1, keepdims=True)) / W.std(1, keepdims=True)).astype(np.float32)

def build_set(records, filt):
    Xs, syms, pids, poss, rrs = [], [], [], [], []
    rrms, hws = [], []          # extensions: pre-RR / record mean RR, Huang windows
    for rec in records:
        sig, atr_sym, atr_sample, fs = A03.load_ecg(os.path.join(DATA_DIR, rec), target_fs=360)
        if filt == "median":
            sig_f, _ = A_sig.median_filters(sig, fs=fs, kernel_size_ms1=200, kernel_size_ms2=600, mode='reflect')
        else:   # the authors' (commented-out) morphological line
            _, sig_f, _, _ = A_sig.morph_filter_baseline(sig, fs=360, k1_ratio=0.20, k2_ratio=0.35, k3_ratio=0.005, mode='reflect')
        df_all = pd.DataFrame({'atr_sym': atr_sym, 'atr_sample': atr_sample})
        df = df_all[df_all.atr_sym.isin(NORMAL_S + SVEB_S + VEB_S)].reset_index(drop=True)
        Xp, _, sp, pp = A03.build_XY(sig_f, df, 0.8, 0.8, fs, NORMAL_S, SVEB_S + VEB_S, patient_id=rec)
        L = int(round(0.8 * fs))
        keep = ((df.atr_sample - L) >= 0) & ((df.atr_sample + L + 1) <= len(sig_f))
        pos = df.atr_sample[keep].to_numpy()
        assert len(pos) == len(Xp) and list(df.atr_sym[keep]) == list(sp)
        beats_all = [s for s, y in zip(atr_sample, atr_sym) if y in BEAT_SYMBOLS]
        Xs.append(Xp); syms += list(sp); pids += list(pp); poss.append(pos); rrs.append(rr_features(beats_all, pos, fs))
        rrms.append(rr_mean_ratio(beats_all, pos, fs)); hws.append(huang_windows(sig, pos))
    X = np.vstack(Xs); pos = np.concatenate(poss); rr = np.vstack(rrs); pid = np.array(pids)
    num = A03.Numerical_Labels(syms)
    X_sh, y_sh, pid_sh = A03.Shuffle_data(X, num, pid, seed=42)
    np.random.seed(42); perm = np.random.permutation(len(X))           # the same permutation Shuffle_data used
    assert np.array_equal(X[perm], X_sh)
    with contextlib.redirect_stdout(io.StringIO()):     # silence the authors' "taking all of them" prints
        Xd, yd, pidd = A_strat.Make_Data_Stratified(X_sh, y_sh, pid_sh, num_data=50000, Type_Class='SuperC')
        idx_d, _, _ = A_strat.Make_Data_Stratified(np.arange(len(X))[:, None], y_sh, pid_sh, num_data=50000, Type_Class='SuperC')
    src = perm[np.asarray(idx_d)[:, 0].astype(int)]
    assert np.array_equal(X[src], Xd), "beat-order tracking failed"
    return {"X": np.asarray(Xd, np.float64), "y": np.asarray(yd, int), "pid": np.asarray(pidd).astype(str),
            "pos": pos[src], "rr": rr[src], "sym": np.array(syms)[src],
            "rr_mean_ratio": np.concatenate(rrms)[src], "huang_beats": np.vstack(hws)[src]}

t0 = time.time()
DATA = {f: {ds: build_set(DS_RECORDS[ds], f) for ds in ("DS1", "DS2")} for f in ("morph", "median")}
print(f"datasets built in {time.time()-t0:.0f}s")
for ds in ("DS1", "DS2"):
    assert np.array_equal(DATA["morph"][ds]["y"], DATA["median"][ds]["y"])
    assert np.array_equal(DATA["morph"][ds]["pid"], DATA["median"][ds]["pid"])

# verify against the authors' released DS1 order and counts
_auth = pd.read_csv(os.path.join(REPO_DIR, "splits/patient_ids_train.csv"))
CHECKS["ds1_matches_authors_order"] = bool(np.array_equal(_auth.patient_id.astype(str).values, DATA["morph"]["DS1"]["pid"])
                                           and np.array_equal(_auth.label.values, DATA["morph"]["DS1"]["y"]))
CHECKS["class_counts"] = {ds: np.bincount(DATA["morph"][ds]["y"], minlength=3).tolist() for ds in ("DS1", "DS2")}
CHECKS["expected_counts"] = {"DS1": [45827, 943, 3787], "DS2": [44216, 1837, 3219]}
print("DS1 order identical to authors' file:", CHECKS["ds1_matches_authors_order"], "| counts:", CHECKS["class_counts"])

# folds: the authors' exact indices, and a check that they regenerate
_fd = json.load(open(os.path.join(REPO_DIR, "splits/fold_definitions.json")))
FOLDS = [{"rep": f["rep"], "fold": f["fold"], "train_idx": np.array(f["train_idx"]), "val_idx": np.array(f["val_idx"])}
         for f in _fd["folds"]]
_ok, _k = True, 0
_y1, _p1 = DATA["morph"]["DS1"]["y"], DATA["morph"]["DS1"]["pid"]
for rep, sd in enumerate([11, 23, 37], 1):
    for fold, (tr, va) in enumerate(StratifiedGroupKFold(4, shuffle=True, random_state=sd).split(np.zeros(len(_y1)), _y1, _p1), 1):
        f = FOLDS[_k]; _k += 1
        _ok &= (f["rep"], f["fold"]) == (rep, fold) and np.array_equal(np.sort(f["train_idx"]), tr) and np.array_equal(np.sort(f["val_idx"]), va)
CHECKS["folds_regenerate_exactly"] = bool(_ok)
print("authors' fold indices regenerate exactly:", _ok)

# smoke test: keep a few beats per record and class, remap folds
if SMOKE_TEST:
    def _keep(D):
        keep = []
        for p in np.unique(D["pid"]):
            for c, k in SMOKE_PER_CLASS.items():
                keep += list(np.where((D["pid"] == p) & (D["y"] == c))[0][:k])
        return np.sort(np.array(keep))
    _k1, _k2 = _keep(DATA["morph"]["DS1"]), _keep(DATA["morph"]["DS2"])
    for f in DATA:
        for ds, kk in (("DS1", _k1), ("DS2", _k2)):
            DATA[f][ds] = {k: v[kk] for k, v in DATA[f][ds].items()}
    _remap_arr = -np.ones(int(_k1.max()) + 1, int); _remap_arr[_k1] = np.arange(len(_k1))
    def _remap(idx):
        r = _remap_arr[idx[idx <= _k1.max()]]
        return r[r >= 0]
    FOLDS = [{"rep": f["rep"], "fold": f["fold"], "train_idx": _remap(f["train_idx"]), "val_idx": _remap(f["val_idx"])}
             for f in FOLDS]
    print("SMOKE subset:", {ds: np.bincount(DATA['morph'][ds]['y'], minlength=3).tolist() for ds in ("DS1", "DS2")})

CV_SPLITS = [(f["rep"], f["fold"]) for f in FOLDS]
if SMOKE_TEST:
    CV_SPLITS = SMOKE_CV_SPLITS
FOLD_BY = {(f["rep"], f["fold"]): f for f in FOLDS}

# ---------------- normalization (two paths, as the authors have them) ----------------
def normalize_cv(X, tr_idx, norm):
    """CV path (script 03 get_fold_data + float32 cast before encoding)."""
    if norm == "standard":
        return np.asarray(A_strat.minmax_per_beat(X), np.float32), {}
    mn, mx = A_strat.robust_minmax_fit(X[tr_idx], low=0.1, high=99.9)
    return np.asarray(A_strat.robust_minmax_apply(X, mn, mx), np.float32), {"p_low": mn, "p_high": mx}

def normalize_final(X1, X2, norm):
    """Final DS1 -> DS2 path (script 01): float64, global DS1 percentiles for robust."""
    if norm == "standard":
        return A_strat.Rowwise_Normalization(X1), A_strat.Rowwise_Normalization(X2), {}
    lo, hi = np.percentile(X1, [0.1, 99.9])
    return A_strat.minmax_apply(X1, lo, hi), A_strat.minmax_apply(X2, lo, hi), {"p_low": float(lo), "p_high": float(hi)}


## 6. Spike encoders, RR channels, measured A_in

In [ ]:
# Encoders (the authors' functions, device passed explicitly), RR spike channels, A_in measurement.
PAPER_AIN = {"multi": 0.05892, "rate": 20.8245, "latency": 0.9473}   # values hard-coded by the authors

def multithreshold_t(Xb, num_levels=200):
    """GPU version of the authors' MultiThreshold (detect_edges='both', include_endpoints=True,
    separate_polarity=False). Same thresholds (np.linspace in the data's dtype) and the same comparisons:
    rise = x_t >= thr & x_{t-1} < thr, fall = x_t < thr & x_{t-1} >= thr, spike = rise | fall."""
    X = np.ascontiguousarray(np.asarray(Xb))
    thr = torch.from_numpy(np.linspace(0.0, 1.0, num_levels, dtype=X.dtype)).to(DEVICE)
    x = torch.from_numpy(X).to(DEVICE)
    out = torch.zeros((X.shape[0], X.shape[1], num_levels), dtype=torch.uint8, device=DEVICE)
    above = x[:, :, None] >= thr                     # (B, T, K)
    out[:, 1:] = (above[:, 1:] != above[:, :-1]).to(torch.uint8)
    return out

def roundrobin_t(counts, bin_pop=10):
    """GPU version of the authors' counts_to_binary_population_roundrobin. Their loop keeps a pointer p per
    (beat, channel), sets sub-channels (p + j) % bin_pop for j < min(k, bin_pop), then p = (p + k) % bin_pop.
    So p_t = (counts before t) % bin_pop, and sub-channel b fires iff (b - p_t) % bin_pop < min(k_t, bin_pop)."""
    c = torch.as_tensor(np.asarray(counts).astype(np.int64), device=DEVICE)          # (B, L, C)
    p = torch.remainder(torch.cumsum(c, dim=1) - c, bin_pop)
    keff = torch.clamp(c, max=bin_pop)
    b = torch.arange(bin_pop, device=DEVICE)
    rel = torch.remainder(b[None, None, None, :] - p[..., None], bin_pop)
    out = (rel < keff[..., None]).to(torch.uint8)
    return out.reshape(c.shape[0], c.shape[1], c.shape[2] * bin_pop)

def encode_t(Xb, enc, num_levels=200):
    """(B, 577) normalized beats -> (B, T_in, C) uint8 spikes as a tensor on DEVICE."""
    if enc == "multi":
        return multithreshold_t(Xb, num_levels)
    if enc == "rate":
        s = A_enc.TemporalRateCoding(Xb, None, gain=1.0, offset=0.0, pop_size=64, batch_size=1024, device=DEVICE,
                                     out_dir=None, spike_filename=None, labels_filename=None, save_to_disk=False)
        return torch.from_numpy(np.asarray(s, dtype=np.uint8)).to(DEVICE)
    if enc == "latency":
        cnt = A_enc.BoundedDelayLatencyCoding(Xb, Dmax=64, pop_size=1, batch_size=1024, device=DEVICE)
        return roundrobin_t(cnt, bin_pop=10)
    raise ValueError(enc)

def encode(Xb, enc, num_levels=200):
    """Same as encode_t, returned as a NumPy array."""
    return encode_t(Xb, enc, num_levels).cpu().numpy()

N_IN = {"multi": 200, "rate": 64, "latency": 10}

# RR timing as extra input channels: thermometer code of 3 ratio features, 10 levels each.
RR_LIQ_FEATS = [RR_NAMES.index(n) for n in ("pre_over_local", "post_over_local", "pre_over_rec")]
RR_LEVELS, RR_PERIOD = 10, 5            # an active channel spikes every 5th step

def rr_liq_feats(rr_rows):
    """Columns turned into RR channels: the 3 ratios; with the 9-column array of rr_inputs(..., True) also the
    9th column, pre-RR / recording mean RR (40 channels instead of 30)."""
    return RR_LIQ_FEATS + ([len(RR_NAMES)] if rr_rows.shape[1] > len(RR_NAMES) else [])

def rr_thresholds(rr_train):
    q = np.linspace(0.05, 0.95, RR_LEVELS)
    return np.stack([np.quantile(rr_train[:, j], q) for j in rr_liq_feats(rr_train)])     # (3 or 4, 10)

def rr_channels(rr_rows, thr, T):
    active = (rr_rows[:, rr_liq_feats(rr_rows)][:, :, None] >= thr[None]).reshape(len(rr_rows), -1)   # (B, 30 or 40)
    pulse = (np.arange(T) % RR_PERIOD == 0).astype(np.uint8)
    return (pulse[None, :, None] * active[:, None, :]).astype(np.uint8)                   # (B, T, 30)

def rr_window_counts(rr_rows, thr, T):
    """rr_channels(rr_rows, thr, T) summed over the 32 windows of 20 steps (steps >= T are silent), (B, 32, 30) uint8."""
    active = (rr_rows[:, rr_liq_feats(rr_rows)][:, :, None] >= thr[None]).reshape(len(rr_rows), -1)
    steps = np.arange(NWIN * WIN)
    per_window = ((steps % RR_PERIOD == 0) & (steps < T)).reshape(NWIN, WIN).sum(1)
    return (per_window[None, :, None] * active[:, None, :]).astype(np.uint8)

def measure_ain(Xn, rows, enc, num_levels=200, rr=None, rr_thr=None, n=5000):
    """Average number of active input spikes per time step (summed over channels), over a random
    subset of training beats. Same quantity as the authors' estimate_K_spikes_per_timestep."""
    rows = np.asarray(rows)
    torch.manual_seed(12345)           # the Rate encoder is stochastic: make A_in independent of job order
    pick = np.sort(np.random.default_rng(0).choice(rows, min(n, len(rows)), replace=False))
    tot, cnt = 0.0, 0
    for i in range(0, len(pick), 1000):
        r = pick[i:i + 1000]
        S = encode(Xn[r], enc, num_levels)
        if rr is not None:
            S = np.concatenate([S, rr_channels(rr[r], rr_thr, S.shape[1])], axis=2)
        tot += float(S.sum(dtype=np.int64)); cnt += S.shape[0] * S.shape[1]
    return tot / cnt


## 7. The paper's liquid class

In [ ]:
# The liquid: built with the authors' class; modes patch the two bugs. GPU simulator with diagnostics.
LIQ = dict(P_in=0.1, betaE=0.95, betaI=0.8, Uthr=1.0, rho=0.9, reservoir_dim=(7, 7, 7), excit_ratio=0.8,
           dt=1.0, num_sim_steps=650, C_vals={"exc_exc": 0.3, "exc_inh": 0.2, "inh_exc": 0.4, "inh_inh": 0.1},
           lambda_val=0.8, scl_v=1, mu_rsv=[0.020, 0.025, 0.050, 0.040], g_rec=1, tau_syn=0.01)
I_TARGET = LIQ["rho"] * (1 - LIQ["betaE"]) * LIQ["Uthr"]
MODES = {   # gamma_fix: Gamma(shape, scale) as intended; ain: "paper" = hard-coded value, "measured" = Eq. as described
    "published": {"gamma_fix": False, "ain": "paper"},
    "fixed":     {"gamma_fix": True,  "ain": "measured"},
    "fix_gamma": {"gamma_fix": True,  "ain": "paper"},
    "fix_ain":   {"gamma_fix": False, "ain": "measured"},
}
_TorchGamma = torch.distributions.Gamma

def build_liquid(input_size, mu_in, seed, gamma_fix, lambda_val=0.8, recurrence=True, g_rec=None):
    """seed_everything(seed) then the authors' constructor, exactly as in their scripts."""
    seed_everything(seed, deterministic=False)
    if gamma_fix:
        torch.distributions.Gamma = lambda conc, scale: _TorchGamma(conc, 1.0 / scale)   # (shape, scale) as intended
    try:
        model = LSM3D(input_size, LIQ["P_in"], mu_in, LIQ["betaE"], LIQ["betaI"], LIQ["reservoir_dim"],
                      LIQ["excit_ratio"], LIQ["dt"], LIQ["num_sim_steps"], LIQ["C_vals"], lambda_val, LIQ["scl_v"],
                      np.array(LIQ["mu_rsv"]), LIQ["g_rec"] if g_rec is None else g_rec, LIQ["tau_syn"], DEVICE).to(DEVICE)
    finally:
        torch.distributions.Gamma = _TorchGamma
    if not recurrence:
        model.rsv_weights = torch.zeros_like(model.rsv_weights)
    return model

def structure_stats(model):
    W = model.rsv_weights.detach().cpu().numpy(); Win = model.input_rsv_weights.detach().cpu().numpy()
    E = model.is_excit.cpu().numpy()
    out = {"n_recurrent_synapses": int((W != 0).sum()), "mean_recurrent_in_degree": float((W != 0).sum(0).mean()),
           "neurons_without_recurrent_input": int(((W != 0).sum(0) == 0).sum()),
           "n_input_synapses": int((Win != 0).sum()), "mean_input_in_degree": float((Win != 0).sum(0).mean()),
           "mean_nonzero_input_weight": float(Win[Win != 0].mean()) if (Win != 0).any() else 0.0,
           "threshold": LIQ["Uthr"], "n_excitatory": int(E.sum())}
    for name, pre, post in (("EE", E, E), ("EI", E, ~E), ("IE", ~E, E), ("II", ~E, ~E)):
        blk = W[np.ix_(pre, post)]
        nz = blk[blk != 0]
        out[f"{name}_synapses"] = int(nz.size)
        out[f"{name}_mean_abs_weight"] = float(np.abs(nz).mean()) if nz.size else 0.0
    return out

def save_reservoir(model, meta):
    rid = stable_hash(meta, 12)
    path = os.path.join(WORK_DIR, "reservoirs", f"{rid}.pt")
    if not os.path.exists(path):
        Win = model.input_rsv_weights.detach().cpu(); W = model.rsv_weights.detach().cpu()
        iw, ir = Win.nonzero().T, W.nonzero().T
        torch.save({"meta": meta, "input_size": Win.shape[0], "n_neurons": W.shape[0],
                    "win_index": iw.to(torch.int16), "win_value": Win[iw[0], iw[1]].float(),
                    "wrec_index": ir.to(torch.int16), "wrec_value": W[ir[0], ir[1]].float(),
                    "is_excit": model.is_excit.cpu(), "beta": torch.where(model.is_excit.cpu(), LIQ["betaE"], LIQ["betaI"]).float(),
                    "structure": structure_stats(model), "torch": torch.__version__, "device": str(DEVICE)}, path)
        disk_add(os.path.getsize(path))
    return rid

WIN, NWIN, N_NEUR = 20, 32, 343          # fr_windows(win=20, stride=20) over 650 steps -> 32 windows

def simulate(model, Xn, enc, batch, num_levels=200, rr=None, rr_thr=None, y=None, enc_seed=0):
    """Encode + run the liquid exactly as the authors' loop does. Returns spike counts per (window, neuron)
    as uint8 (N, 32*343) in the authors' feature order (window-major), plus liquid diagnostics."""
    torch.manual_seed(enc_seed)
    N = len(Xn)
    out = np.empty((N, NWIN * N_NEUR), np.uint8)
    Win, Wr = model.input_rsv_weights, model.rsv_weights
    neur = torch.zeros(N_NEUR, dtype=torch.float64, device=DEVICE)
    per_beat = np.empty(N, np.float32)
    acc = {"in_spikes": 0.0, "in_steps": 0, "Iin_sum": 0.0, "Iin_steps": 0, "Irec_abs_sum": 0.0}
    t_enc = t_sim = 0.0
    with torch.no_grad():
        for s in range(0, N, batch):
            t0 = time.time()
            x = encode_t(Xn[s:s + batch], enc, num_levels)
            if rr is not None:
                x = torch.cat([x, torch.from_numpy(rr_channels(rr[s:s + batch], rr_thr, x.shape[1])).to(DEVICE)], dim=2)
            if DEVICE.type == "cuda":
                torch.cuda.synchronize()
            t_enc += time.time() - t0; t0 = time.time()
            B, Tin, C = x.shape
            spk = torch.zeros(B, N_NEUR, device=DEVICE); mem = torch.zeros(B, N_NEUR, device=DEVICE)
            zero_in = torch.zeros(B, C, device=DEVICE)
            cnt = torch.zeros(B, NWIN, N_NEUR, dtype=torch.int16, device=DEVICE)
            Iin_acc = torch.zeros((), dtype=torch.float64, device=DEVICE); Irec_acc = torch.zeros((), dtype=torch.float64, device=DEVICE)
            for t in range(LIQ["num_sim_steps"]):
                x_t = x[:, t, :].float() if t < Tin else zero_in
                I_in = x_t @ Win
                I_rec = spk @ Wr
                I = (I_in + I_rec) * model.dt
                spk, mem = model.lif(I, mem)
                if t < NWIN * WIN:
                    cnt[:, t // WIN] += spk.to(torch.int16)
                if t < Tin:
                    Iin_acc += I_in.mean(dtype=torch.float64)
                Irec_acc += I_rec.abs().mean(dtype=torch.float64)
            c = cnt.reshape(B, -1)
            out[s:s + B] = c.to(torch.uint8).cpu().numpy()
            neur += cnt.sum((0, 1), dtype=torch.float64)
            per_beat[s:s + B] = c.sum(1).float().cpu().numpy()
            acc["in_spikes"] += float(x.sum(dtype=torch.int64)); acc["in_steps"] += B * Tin
            acc["Iin_sum"] += float(Iin_acc) * B; acc["Iin_steps"] += Tin * B
            acc["Irec_abs_sum"] += float(Irec_acc) * B
            if DEVICE.type == "cuda":
                torch.cuda.synchronize()
            t_sim += time.time() - t0
            del x, cnt, spk, mem
    neur = neur.cpu().numpy()
    steps = NWIN * WIN
    E = model.is_excit.cpu().numpy()
    stats = {"liquid_firing_prob_per_step": float(neur.sum() / (N * steps * N_NEUR)),
             "excitatory_firing_prob": float(neur[E].sum() / (N * steps * E.sum())),
             "inhibitory_firing_prob": float(neur[~E].sum() / (N * steps * (~E).sum())),
             "silent_neurons": int((neur == 0).sum()), "silent_fraction": float((neur == 0).mean()),
             "liquid_spikes_per_beat_mean": float(per_beat.mean()),
             "input_spikes_per_step": acc["in_spikes"] / max(acc["in_steps"], 1),
             "mean_input_current_per_neuron_step": acc["Iin_sum"] / max(acc["Iin_steps"], 1),
             "input_current_over_target": (acc["Iin_sum"] / max(acc["Iin_steps"], 1)) / I_TARGET,
             "mean_abs_recurrent_current": acc["Irec_abs_sum"] / max(N * LIQ["num_sim_steps"], 1),
             "n_beats": N, "encode_seconds": t_enc, "simulate_seconds": t_sim}
    if y is not None:
        for c, nm in enumerate(CLASS_NAMES):
            if (y == c).any():
                stats[f"liquid_spikes_per_beat_{nm}"] = float(per_beat[y == c].mean())
    return out, stats

def liquid_raster(model, S, batch=256):
    """Full spike raster (B, 650, 343) with the same update as simulate(), on the device. Also returns the mean
    |recurrent current| and mean input current per neuron and step (input steps only)."""
    Win, Wr = model.input_rsv_weights, model.rsv_weights
    T = LIQ["num_sim_steps"]; out = np.zeros((len(S), T, N_NEUR), bool); irec = iin = 0.0; tin = 1
    with torch.no_grad():
        for s in range(0, len(S), batch):
            x = torch.from_numpy(np.ascontiguousarray(S[s:s + batch])).to(DEVICE).float()
            B, tin, Cin = x.shape
            spk = torch.zeros(B, N_NEUR, device=DEVICE); mem = torch.zeros(B, N_NEUR, device=DEVICE)
            zero_in = torch.zeros(B, Cin, device=DEVICE)
            r = torch.zeros(B, T, N_NEUR, dtype=torch.bool, device=DEVICE)
            for t in range(T):
                I_in = (x[:, t, :] if t < tin else zero_in) @ Win
                I_rec = spk @ Wr
                spk, mem = model.lif((I_in + I_rec) * model.dt, mem)
                r[:, t] = spk > 0
                irec += float(I_rec.abs().mean()) * B
                if t < tin:
                    iin += float(I_in.mean()) * B
            out[s:s + B] = r.cpu().numpy()
    return out, irec / (len(S) * T), iin / (len(S) * tin)

def counts_to_rates(counts_rows):
    """Authors' features: window mean of 0/1 spikes in float64, stored float32."""
    return (counts_rows.astype(np.float64) / WIN).astype(np.float32)


## 8. Replication functions reused

In [ ]:
# From the replication notebook: the split names (verbatim), and its readout (verbatim, except that the power
# of the class weights is a parameter; 0.6, the paper's, is the default and gives the same fit).
def parse_split(s):
    m = re.match(r"cv-r(\d+)f(\d+)", s)
    return (int(m.group(1)), int(m.group(2))) if m else None

def fit_readout_p(Xtr_fn, ntr, ytr, Xev_fn, nev, loss, rng_seed, yev_for_curve=None, curve_sub=4000, prescale=False, n_jobs=None,
                  prepared=None, cancel=None, epochs=None, weight_power=0.6):
    """Authors' readout: StandardScaler (batched partial_fit) + SGDClassifier(hinge|log_loss, l2, alpha=1e-4,
    optimal LR, average=True, random_state=2026, soft class weights ^0.6), 15 epochs, batch 4096,
    shuffling with np.random.default_rng(rng_seed). Batches are scaled on the fly as in the authors' code, or, when
    prescale=True and RAM allows, the whole matrix is scaled once (row-wise identical values, faster).
    prepared=(scaler, XS) reuses a scaler / scaled matrix shared by several fits on the same training rows.
    Xtr_fn(positions) / Xev_fn(positions) return float32 feature rows."""
    bs = 4096
    scaler, XS = prepared if prepared is not None else prepare_train(Xtr_fn, ntr, prescale)
    tf = lambda X: scaler.transform(X).astype(np.float32)
    get_tr = (lambda p: XS[p]) if XS is not None else (lambda p: tf(Xtr_fn(p)))
    clf = SGDClassifier(loss=loss, penalty="l2", alpha=1e-4, fit_intercept=True, learning_rate="optimal",
                        max_iter=1, tol=None, class_weight=soft_class_weight(ytr, weight_power), average=True, random_state=2026,
                        n_jobs=n_jobs)   # n_jobs only parallelises the 3 one-vs-rest problems: results are bit-identical
    classes = np.unique(ytr)
    rng = np.random.default_rng(rng_seed)
    sub = np.sort(np.random.default_rng(0).choice(ntr, min(curve_sub, ntr), replace=False))
    Xsub = get_tr(sub)
    def predict_eval():
        preds, scores = [], []
        for i in range(0, nev, bs):
            Xe = tf(Xev_fn(np.arange(i, min(i + bs, nev))))
            preds.append(clf.predict(Xe)); scores.append(clf.decision_function(Xe).astype(np.float32))
        return np.concatenate(preds), np.concatenate(scores)
    curve = []
    for ep in range(epochs or READOUT_EPOCHS):          # epochs: only the core search uses fewer
        idx = rng.permutation(ntr)
        for i in range(0, ntr, bs):
            if cancel is not None and cancel.is_set():
                raise Cancelled()
            sel = idx[i:i + bs]
            Xb = get_tr(sel)
            if ep == 0 and i == 0:
                clf.partial_fit(Xb, ytr[sel], classes=classes)
            else:
                clf.partial_fit(Xb, ytr[sel])
        row = {"epoch": ep + 1, "train_sub_macro_f1": compute_metrics(ytr[sub], clf.predict(Xsub))["macro_f1"]}
        if loss == "hinge":
            dec = clf.decision_function(Xsub)
            onehot = -np.ones_like(dec); onehot[np.arange(len(sub)), ytr[sub]] = 1
            row["train_sub_hinge_loss"] = float(np.maximum(0, 1 - onehot * dec).mean())
        if yev_for_curve is not None and (ep + 1) in CURVE_EVAL_EPOCHS:
            row["eval_macro_f1"] = compute_metrics(yev_for_curve, predict_eval()[0])["macro_f1"]
        curve.append(row)
    y_pred, scores = predict_eval()
    return scaler, clf, y_pred, scores, curve


## 9. Shared task code (the fixed liquid, features, readout, statistics, the check)

In [ ]:
# Shared by the four task notebooks.
# The liquid is the paper's with the three fixes: Gamma weights drawn as the paper describes, A_in measured on each
# split's training rows (Eq. 12-13 as described), and the reset of Eqs. 17-19 (lsm2's simulator; the authors' snnTorch
# neuron subtracts once more while the stored voltage stays above 1). Everything else is config 11: morphological
# filter, robust normalisation, Multi-Threshold coding (200 levels), 7x7x7 liquid, windowed firing rates, the authors'
# linear SVM readout, the authors' splits (DS2 and the 3 x 4 grouped CV of DS1) and readout seeds.
import threading
from concurrent.futures import ThreadPoolExecutor
from lsm2 import config as LC, liquid as LL, sim as LS, stats as LSt

LS.USE_CUDA_GRAPHS = LS.USE_CUDA_GRAPHS and DEVICE.type == "cuda"
ENC = "multi"
TASK_DIR = os.path.join(WORK_DIR, TASK)
os.makedirs(TASK_DIR, exist_ok=True)
SFX = "_smoke" if SMOKE_TEST else ""
SPLITS_CV = [f"cv-r{r}f{f}" for r, f in CV_SPLITS]
ALL_SPLITS = ["final"] + SPLITS_CV
D1, D2 = DATA["morph"]["DS1"], DATA["morph"]["DS2"]

def tpath(name):
    return os.path.join(TASK_DIR, f"{name}{SFX}")

def geometry(n_samples):
    """(windows, steps) for an input of n_samples steps, by the paper's rule: 577 samples -> 32 windows of 20 steps,
    650 steps simulated (the windows cover the input, the last 10 steps are simulated but not read)."""
    nwin = math.ceil(n_samples / WIN) + 3
    return nwin, nwin * WIN + 10

assert geometry(D1["X"].shape[1]) == (NWIN, LIQ["num_sim_steps"]), "window rule does not give the paper's 32 x 20 / 650"


# ---------------------------------------------------------------- front end and splits (the replication's)
def front_end(split, cast, X1=None, X2=None, folds=None):
    """Robust normalisation for one split. cast=True: the replication's feature path (float32 also for the final
    split); cast=False: its liquid path (the final split stays float64, as execute_lsm passes it to the encoder)."""
    X1 = D1["X"] if X1 is None else X1
    X2 = D2["X"] if X2 is None else X2
    folds = FOLD_BY if folds is None else folds
    if split == "final":
        X1n, X2n, normp = normalize_final(X1, X2, "robust")
        if cast:
            X1n, X2n = np.asarray(X1n, np.float32), np.asarray(X2n, np.float32)
        return X1n, X2n, np.arange(len(X1)), normp
    f = folds[parse_split(split)]
    X1n, normp = normalize_cv(X1, f["train_idx"], "robust")
    return X1n, None, f["train_idx"], normp

def split_rows(split, n1, n2, folds=None):
    """Training rows (DS1), evaluation rows (DS2 for the final split, DS1 otherwise), readout seed (the authors')."""
    if split == "final":
        return np.arange(n1), np.arange(n2), 2026
    rep, fold = parse_split(split)
    f = (FOLD_BY if folds is None else folds)[(rep, fold)]
    return f["train_idx"], f["val_idx"], 2026 + rep * 10 + fold


# ---------------------------------------------------------------- features
def window_counts(Xn, nwin, batch=2048):
    """The replication's encoder_window_counts with nwin windows: Multi-Threshold spikes counted per 20 steps."""
    C = 200
    out = np.zeros((len(Xn), nwin, C), np.uint8)
    for s in range(0, len(Xn), batch):
        S = encode(Xn[s:s + batch], ENC)
        T = min(S.shape[1], nwin * WIN)
        Sp = np.zeros((S.shape[0], nwin * WIN, C), np.uint8); Sp[:, :T] = S[:, :T]
        out[s:s + batch] = Sp.reshape(S.shape[0], nwin, WIN, C).sum(2)
    return out

def rr_counts(rr_rows, thr, T_in, nwin):
    """The replication's rr_window_counts with nwin windows (the RR channels, counted per window)."""
    active = (rr_rows[:, rr_liq_feats(rr_rows)][:, :, None] >= thr[None]).reshape(len(rr_rows), -1)
    steps = np.arange(nwin * WIN)
    per_window = ((steps % RR_PERIOD == 0) & (steps < T_in)).reshape(nwin, WIN).sum(1)
    return (per_window[None, :, None] * active[:, None, :]).astype(np.uint8)

def liquid_model(X1n, tr_rows, seed, rr1=None, rr_thr=None):
    """The paper's liquid (the authors' class) with the fixed input rule: A_in measured on the training rows,
    Gamma weights as described. With RR, the 30 RR channels are extra inputs (and count in A_in)."""
    ain = measure_ain(X1n, tr_rows, ENC, rr=rr1, rr_thr=rr_thr)
    mu_in = I_TARGET / (ain * LIQ["P_in"])
    n_in = 200 + (len(rr_thr) * RR_LEVELS if rr_thr is not None else 0)
    return build_liquid(n_in, mu_in, seed, True), float(ain), float(mu_in)

def simulate_x(model, Xn, T, nwin, rr=None, rr_thr=None, recurrence=True, quirk=False):
    """Window spike counts (n, nwin * 343), window-major as the replication stores them. lsm2's simulator on the
    authors' weights, voltages start at 0. quirk=True: the authors' snnTorch reset (used only by the check that
    this path reproduces the replication's stored runs); quirk=False: Eqs. 17-19."""
    liq = LL.from_authors_model(model, snn_quirk=bool(quirk))
    sm = LS.Simulator(liq, device=DEVICE, dtype=torch.float32, g=1.0, in_scale=1.0, recurrence=recurrence)
    n, N = len(Xn), liq.N
    out = np.zeros((n, nwin * N), np.uint8)
    spikes, t0 = 0.0, time.time()
    for s in range(0, n, NEW_SIM_BATCH):
        x = encode_t(Xn[s:s + NEW_SIM_BATCH], ENC)
        if rr is not None:
            x = torch.cat([x, torch.from_numpy(rr_channels(rr[s:s + NEW_SIM_BATCH], rr_thr, x.shape[1])).to(DEVICE)], dim=2)
        o = sm.run(x, np.zeros((len(x), N)), T, rec={"windows": (WIN, nwin)})
        w = o["windows"]
        spikes += float(w.sum())
        out[s:s + len(x)] = w.clamp(max=255).to(torch.uint8).reshape(len(x), -1).cpu().numpy()
    rate = spikes / (n * N * nwin * WIN * LC.SAMPLE_MS / 1000.0)
    return out, {"rate_hz": rate, "sim_seconds": time.time() - t0}

def rates_fn(C):
    """Features of a counts array (n, ...): firing rate per window and neuron (counts / 20), float32."""
    return lambda r: counts_to_rates(C[r].reshape(len(r), -1))

def elm_fns(c1, c2, model, tr_rows):
    """The replication's random-features control: window counts x the liquid's input weights, centred on the
    training mean, ReLU (no time dynamics, no recurrence)."""
    Wf = model.input_rsv_weights.detach().cpu().numpy().astype(np.float32)
    nwin = c1.shape[1]
    proj = lambda c: np.einsum("nwc,ck->nwk", c.astype(np.float32), Wf).reshape(len(c), -1)
    tot = np.zeros(nwin * N_NEUR, np.float64)
    for i in range(0, len(tr_rows), 4096):
        tot += proj(c1[tr_rows[i:i + 4096]]).sum(0)
    mean = (tot / len(tr_rows)).astype(np.float32)
    F1 = lambda r: np.maximum(proj(c1[r]) - mean, 0)
    F2 = (lambda r: np.maximum(proj(c2[r]) - mean, 0)) if c2 is not None else None
    return F1, F2


# ---------------------------------------------------------------- readout and the fit queue
def readout(F_tr, tr, y_tr_all, F_ev, ev, y_ev_all, rng_seed, weight_power=0.6, prepared=None):
    """The authors' readout (linear SVM) on rows tr -> rows ev. F_tr / F_ev map row indices to features."""
    ftr = lambda p: F_tr(tr[p]); fev = lambda p: F_ev(ev[p])
    _, _, yp, sc, _ = fit_readout_p(ftr, len(tr), y_tr_all[tr], fev, len(ev), "hinge", rng_seed, prescale=PRESCALE,
                                    n_jobs=1, prepared=prepared, weight_power=weight_power)
    m = compute_metrics(y_ev_all[ev], yp)
    return {k: v for k, v in m.items() if k != "confusion_matrix"}, m.get("confusion_matrix"), yp, sc

def prepare(F_tr, tr):
    """A scaler (and the scaled training matrix) shared by several fits on the same training rows."""
    return prepare_train(lambda p: F_tr(tr[p]), len(tr), PRESCALE)

class FitQueue:
    """Readout fits run side by side (FIT_THREADS); one row per fit in TASK_DIR/rows.json, saved after every fit,
    so a restarted session skips what is done."""
    def __init__(self, keys):
        self.keys, self.path = keys, tpath("rows") + ".json"
        self.rows = json.load(open(self.path)) if os.path.exists(self.path) else []
        self.done = {self.key(r) for r in self.rows}
        self.lock = threading.Lock()
        self.ex = ThreadPoolExecutor(max(1, FIT_THREADS))
        self.futs = []
    def key(self, r):
        return tuple(r.get(k) for k in self.keys)
    def is_done(self, **meta):
        return self.key(meta) in self.done
    def submit(self, meta, job):
        if self.is_done(**meta):
            return
        def run():
            t0 = time.time()
            m, cm, _, _ = job()
            row = {**meta, **m, "confusion_matrix": cm, "fit_seconds": time.time() - t0, "session": SESSION_TAG}
            with self.lock:
                self.rows.append(row); self.done.add(self.key(row))
                jdump(self.rows, self.path)
            log("    " + ", ".join(f"{k} {meta[k]}" for k in self.keys) +
                f": Macro-F1 {m['macro_f1']:.4f}, SVEB recall {m['recall_SVEB']:.3f}, SVEB F1 {m['f1_SVEB']:.3f}, "
                f"VEB recall {m['recall_VEB']:.3f} ({time.time() - t0:.0f} s)")
        self.futs.append(self.ex.submit(run))
    def wait(self, max_pending=0):
        """Block until at most max_pending fits are pending (keeps at most a few states in memory)."""
        while True:
            for f in [f for f in self.futs if f.done()]:
                f.result()                                   # re-raises a failed fit here
                self.futs.remove(f)
            if len(self.futs) <= max_pending:
                return
            time.sleep(0.5)

def budget_left():
    if time.time() - SESSION_START > SESSION_BUDGET_HOURS * 3600:
        log(f"{TASK}: session budget reached; run the notebook again to continue")
        return False
    return True


# ---------------------------------------------------------------- comparisons and reporting
def as_runs(df):
    """A table in lsm2.stats' format (spec_split, spec_seed, metrics)."""
    if df is None or len(df) == 0:
        return None
    d = df.rename(columns={"split": "spec_split", "seed": "spec_seed"}).copy()
    d["spec_seed"] = pd.to_numeric(d["spec_seed"], errors="coerce")
    return d

def compare_family(name, items):
    """items: [(label, table A, table B)] -> lsm2.stats' paired comparisons (Nadeau-Bengio over the CV splits,
    seeds averaged per split; DS2 difference by seed), Holm within the family, verdicts."""
    res = [(lab, LSt.compare(as_runs(a), as_runs(b), require_seed_sign=True) if a is not None and b is not None and len(a) and len(b) else None)
           for lab, a, b in items]
    return pd.DataFrame(LSt.family(name, res))

def means_table(groups):
    """groups: [(label, table)] -> mean Macro-F1 and per-class recall / F1, over the CV runs and over the DS2 runs."""
    rows = []
    for lab, df in groups:
        if df is None or len(df) == 0:
            rows.append({"model": lab, "cv_runs": 0, "ds2_runs": 0}); continue
        cv = df[df.split.astype(str).str.startswith("cv-")]; f = df[df.split == "final"]
        r = {"model": lab, "cv_runs": int(len(cv)), "ds2_runs": int(len(f))}
        for k in ("macro_f1", "recall_SVEB", "f1_SVEB", "precision_SVEB", "recall_VEB", "f1_VEB"):
            r[f"cv_{k}"] = float(cv[k].mean()) if len(cv) else np.nan
            r[f"ds2_{k}"] = float(f[k].mean()) if len(f) else np.nan
        rows.append(r)
    return pd.DataFrame(rows)

def md_table(df, cols, fmt=None):
    fmt = fmt or {}
    cols = [c for c in cols if c in df.columns]
    lines = ["| " + " | ".join(cols) + " |", "|" + "---|" * len(cols)]
    for _, r in df.iterrows():
        cells = []
        for c in cols:
            v = r.get(c)
            if isinstance(v, (float, np.floating)):
                cells.append("" if not np.isfinite(v) else fmt.get(c, "{:.4f}").format(v))
            else:
                cells.append("" if v is None or (isinstance(v, float) and np.isnan(v)) else str(v))
        lines.append("| " + " | ".join(cells) + " |")
    return "\n".join(lines)

MEANS_COLS = ["model", "cv_runs", "cv_macro_f1", "cv_recall_SVEB", "cv_f1_SVEB", "cv_recall_VEB", "ds2_runs",
              "ds2_macro_f1", "ds2_recall_SVEB", "ds2_f1_SVEB", "ds2_precision_SVEB", "ds2_recall_VEB"]
COMP_COLS = ["comparison", "n_splits", "cv_diff", "ci_low", "ci_high", "p_holm", "ds2_diff", "ds2_diff_recall_SVEB", "verdict"]

def log_comparisons(C):
    for _, r in C.iterrows():
        if pd.notna(r.get("cv_diff")):
            log(f"{TASK}: {r['comparison']}: CV {r['cv_diff']:+.4f} [{r['ci_low']:+.4f}, {r['ci_high']:+.4f}], "
                f"DS2 {r['ds2_diff']:+.4f}: {r['verdict']}")
        else:
            log(f"{TASK}: {r['comparison']}: missing")


# ---------------------------------------------------------------- check against the replication's stored runs
def stored_runs():
    """Config 11's stored runs of REPLICATION_DIR that the check uses (fixed-mode liquid, random features, no liquid)."""
    if not REPLICATION_DIR:
        return None
    root = os.path.expanduser(REPLICATION_DIR)
    if not os.path.isdir(os.path.join(root, "runs")):
        log(f"{TASK} check: no runs/ folder in {root}; the check is skipped")
        return None
    rows = []
    for d in sorted(glob.glob(os.path.join(root, "runs", "*"))):
        try:
            c = json.load(open(os.path.join(d, "config.json")))
        except Exception:
            continue
        s = c.get("spec", {})
        if s.get("filter") != "morph" or s.get("norm") != "robust" or s.get("encoder") != ENC or s.get("rr_set") is not None:
            continue
        lsm_ok = (s.get("kind") == "lsm" and c.get("group") in ("main_cv", "main_final") and s.get("mode") == "fixed"
                  and s.get("readout") == "svm" and not s.get("rr_in_liquid") and s.get("recurrence") is True
                  and s.get("gain") == 1 and s.get("lambda_val") == 0.8 and s.get("ain_override") is None)
        feat_ok = s.get("kind") == "feat" and s.get("model") in ("elm", "no_liquid") and c.get("group") in ("ablation_cv", "ablation_final")
        if not (lsm_ok or feat_ok):
            continue
        try:
            m = json.load(open(os.path.join(d, "metrics.json")))
        except Exception:
            continue
        rows.append({"run_dir": d, "model": "liquid" if lsm_ok else s["model"], "seed": s.get("seed"),
                     "split": s.get("split"), "macro_f1": m.get("macro_f1")})
    return pd.DataFrame(rows) if rows else None

def check_against_replication():
    """Recompute three stored runs of the replication (first seed, first CV split) through this notebook's code:
    the liquid with the authors' reset (quirk), random features, no liquid. Same predictions, beat by beat, and the
    same decision scores mean the front end, A_in, weights, simulator, features and readout are the replication's;
    the task's runs then differ from it only by the reset of Eqs. 17-19."""
    p = tpath("check") + ".json"
    if os.path.exists(p):
        v = json.load(open(p)); log(f"{TASK} check (saved): {v['verdict']}"); return v
    R = stored_runs()
    seed, split = TASK_SEEDS[0], SPLITS_CV[0]
    out = {"seed": seed, "split": split, "checks": []}
    if R is None:
        out["verdict"] = "not run (no REPLICATION_DIR)"; jdump(out, p); log(f"{TASK} check: {out['verdict']}"); return out
    tr, ev, rs = split_rows(split, len(D1["y"]), len(D2["y"]))
    X1l, _, trl, _ = front_end(split, cast=False)
    X1f, _, _, _ = front_end(split, cast=True)
    c1 = window_counts(X1f, NWIN)
    def feats(model):
        if model == "liquid":
            mdl, _, _ = liquid_model(X1l, trl, seed)
            C1, _ = simulate_x(mdl, X1l, LIQ["num_sim_steps"], NWIN, quirk=True)
            return rates_fn(C1)
        if model == "elm":
            mdl, _, _ = liquid_model(X1f, tr, seed)
            return elm_fns(c1, None, mdl, tr)[0]
        return rates_fn(c1)
    for model in ("liquid", "elm", "no_liquid"):
        st = R[(R.model == model) & (R.split == split) & ((R.seed == seed) if model != "no_liquid" else R.seed.isna())]
        if st.empty:
            out["checks"].append({"what": model, "status": "no stored run"}); continue
        F = feats(model)
        m, _, yp, sc = readout(F, tr, D1["y"], F, ev, D1["y"], rs)
        z = np.load(os.path.join(st.run_dir.iloc[0], "predictions.npz"))
        same = float(np.mean(z["y_pred"].astype(int) == np.asarray(yp).astype(int))) if len(z["y_pred"]) == len(yp) else 0.0
        dmax = float(np.abs(z["scores"].astype(np.float64) - np.asarray(sc, np.float64)).max()) if same == 1.0 else np.nan
        ok = same == 1.0 and dmax < 1e-4
        out["checks"].append({"what": model, "status": "match" if ok else "differs", "same_predictions": same,
                              "largest_score_difference": dmax, "macro_f1": m["macro_f1"],
                              "stored_macro_f1": float(st.macro_f1.iloc[0])})
        log(f"{TASK} check: {model}: {'match' if ok else 'differs'} (same predictions {100 * same:.2f}%, "
            f"Macro-F1 {m['macro_f1']:.6f} vs stored {float(st.macro_f1.iloc[0]):.6f})")
    st_ = [c["status"] for c in out["checks"]]
    out["verdict"] = ("all match" if st_ and all(s == "match" for s in st_) else
                      "not run (no stored runs)" if all(s == "no stored run" for s in st_) else
                      "DIFFERS: see check.json (a GPU of another type can flip a few spikes; a large difference is a bug)")
    log(f"{TASK} check: {out['verdict']}")
    jdump(out, p)
    return out


## 10. Task 4: a longer window (the neighbouring beats) and the ladder

In [ ]:
# Task 4: does a longer window (the neighbouring beats, so their timing) help, and does the liquid use it?
# Each beat is cut at R +- WINDOW_HALF_S seconds from the same filtered records (the paper: +- 0.8 s = 577 samples).
# Every window size uses the same beats (those whose longest window fits inside the record) and the authors' folds
# restricted to them. The ladder, for each window size: no liquid, random features, the liquid without its internal
# connections, the full liquid. Windows of 20 steps cover the whole input (the paper's rule: 577 -> 32 windows, 650 steps).
Q = FitQueue(["model", "seed", "split", "half_s"])
W_MODELS = WINDOW_MODELS

def cut_windows(ds, half_s):
    """(beats, 2L + 1) windows of the morphologically filtered record around each beat's R peak, L = round(half_s * 360),
    and which beats fit inside their record (the replication's build_set: the same loader, filter and cut)."""
    D = DATA["morph"][ds]
    L = int(round(half_s * 360))
    X = np.zeros((len(D["y"]), 2 * L + 1), np.float64); ok = np.zeros(len(D["y"]), bool)
    for rec in np.unique(D["pid"]):
        sig, _, _, fs = A03.load_ecg(os.path.join(DATA_DIR, rec), target_fs=360)
        _, sig_f, _, _ = A_sig.morph_filter_baseline(sig, fs=360, k1_ratio=0.20, k2_ratio=0.35, k3_ratio=0.005, mode='reflect')
        idx = np.where(D["pid"] == rec)[0]
        pos = D["pos"][idx]
        good = (pos - L >= 0) & (pos + L + 1 <= len(sig_f))
        for i, p in zip(idx[good], pos[good]):
            X[i] = sig_f[p - L:p + L + 1]
        ok[idx[good]] = True
    return X, ok

def t4_data():
    """Windows per size, the common beats, and the folds restricted to them (indices remapped)."""
    W, ok1, ok2 = {}, np.ones(len(D1["y"]), bool), np.ones(len(D2["y"]), bool)
    for h in WINDOW_HALF_S:
        X1, k1 = cut_windows("DS1", h); X2, k2 = cut_windows("DS2", h)
        W[h] = (X1, X2); ok1 &= k1; ok2 &= k2
    # the 0.8 s cut must be the replication's beats, sample for sample
    if 0.8 in W:
        a1, a2 = W[0.8]
        assert np.array_equal(a1, D1["X"]) and np.array_equal(a2, D2["X"]), "the 0.8 s cut differs from the replication's beats"
    new = -np.ones(len(ok1), int); new[ok1] = np.arange(int(ok1.sum()))
    folds = {}
    for k, f in FOLD_BY.items():
        tr, va = f["train_idx"], f["val_idx"]
        folds[k] = {"rep": f["rep"], "fold": f["fold"], "train_idx": new[tr[ok1[tr]]], "val_idx": new[va[ok1[va]]]}
    W = {h: (X1[ok1], X2[ok2]) for h, (X1, X2) in W.items()}
    return W, ok1, ok2, folds

log(f"{TASK}: start")
CHECK = check_against_replication() if VERIFY else None
WX, OK1, OK2, FOLDS_W = t4_data()
Y1, Y2 = D1["y"][OK1], D2["y"][OK2]
log(f"{TASK}: windows {', '.join(f'+-{h:g} s ({WX[h][0].shape[1]} samples, {geometry(WX[h][0].shape[1])[0]} windows)' for h in WINDOW_HALF_S)}; "
    f"common beats DS1 {int(OK1.sum())} of {len(OK1)} (dropped by class: {np.bincount(D1['y'][~OK1], minlength=3).tolist()}), "
    f"DS2 {int(OK2.sum())} of {len(OK2)} (dropped: {np.bincount(D2['y'][~OK2], minlength=3).tolist()})")

def t4_split(split, h):
    final = split == "final"
    seeds_of = lambda m: (None,) if m == "no_liquid" else TASK_SEEDS
    if all(Q.is_done(model=m, seed=s, split=split, half_s=h) for m in W_MODELS for s in seeds_of(m)):
        return
    X1, X2 = WX[h]
    nwin, T = geometry(X1.shape[1])
    log(f"  {TASK}: {split}, +-{h:g} s ({nwin} windows, {T} steps)")
    tr, ev, rs = split_rows(split, len(Y1), len(Y2), FOLDS_W)
    Dy_ev = Y2 if final else Y1
    X1l, X2l, trl, _ = front_end(split, False, X1, X2, FOLDS_W)
    X1f, X2f, _, _ = front_end(split, True, X1, X2, FOLDS_W)
    c1 = window_counts(X1f, nwin); c2 = window_counts(X2f, nwin) if final else None
    def submit(model, seed, F1, F2):
        Fev = F2 if final else F1
        Q.submit({"model": model, "seed": seed, "split": split, "half_s": h},
                 lambda: readout(F1, tr, Y1, Fev, ev, Dy_ev, rs))
    if "no_liquid" in W_MODELS and not Q.is_done(model="no_liquid", seed=None, split=split, half_s=h):
        submit("no_liquid", None, rates_fn(c1), rates_fn(c2) if final else None)
    for seed in TASK_SEEDS:
        if "random_features" in W_MODELS and not Q.is_done(model="random_features", seed=seed, split=split, half_s=h):
            mdl, _, _ = liquid_model(X1f, tr, seed)
            F1, F2 = elm_fns(c1, c2, mdl, tr)
            submit("random_features", seed, F1, F2)
        need = [m for m in ("liquid", "liquid_norec") if m in W_MODELS and not Q.is_done(model=m, seed=seed, split=split, half_s=h)]
        if need:
            mdl, ain, mu = liquid_model(X1l, trl, seed)       # one liquid: the no-recurrence rung only switches its
            for m in need:                                   # internal connections off (same input weights)
                C1, st = simulate_x(mdl, X1l, T, nwin, recurrence=(m == "liquid"))
                C2 = simulate_x(mdl, X2l, T, nwin, recurrence=(m == "liquid"))[0] if final else None
                log(f"    simulated {m} seed {seed} {split} +-{h:g} s: A_in {ain:.4f}, firing {st['rate_hz']:.2f} Hz, {st['sim_seconds']:.0f} s")
                Q.wait(max_pending=FIT_THREADS)
                submit(m, seed, rates_fn(C1), rates_fn(C2) if final else None)
    Q.wait(max_pending=FIT_THREADS)

for _sp in ALL_SPLITS:
    for _h in WINDOW_HALF_S:
        if not budget_left():
            break
        t4_split(_sp, _h)
        gc.collect()
Q.wait(0)

# ---------------------------------------------------------------- results
R4 = pd.DataFrame(Q.rows)
LAB4 = {"liquid": "liquid", "liquid_norec": "liquid, no internal connections", "random_features": "random features", "no_liquid": "no liquid"}
tab = lambda m, h: R4[(R4.model == m) & (R4.half_s == h)] if len(R4) else None

def diff_table(a, b):
    """Per (split, seed): a - b for the metrics (b seed-free is matched to every seed of a)."""
    if a is None or b is None or not len(a) or not len(b):
        return None
    ks = ["macro_f1", "recall_SVEB", "f1_SVEB", "recall_VEB"]
    bb = b.set_index("split")[ks] if b.seed.isna().all() else b.set_index(["split", "seed"])[ks]
    out = a[["split", "seed"] + ks].copy()
    for i, r in out.iterrows():
        key = r.split if b.seed.isna().all() else (r.split, r.seed)
        if key not in bb.index:
            out.loc[i, ks] = np.nan; continue
        out.loc[i, ks] = r[ks].values - bb.loc[key].values
    return out.dropna(subset=["macro_f1"])

K4 = means_table([(f"{LAB4[m]}, +-{h:g} s", tab(m, h)) for h in WINDOW_HALF_S for m in W_MODELS])
short, long_ = min(WINDOW_HALF_S), max(WINDOW_HALF_S)
items = [(f"{LAB4[m]}: +-{long_:g} s - +-{short:g} s (does the longer window help?)", tab(m, long_), tab(m, short)) for m in W_MODELS]
for h in WINDOW_HALF_S:
    items += [(f"+-{h:g} s: liquid - random features", tab("liquid", h), tab("random_features", h)),
              (f"+-{h:g} s: liquid - no internal connections", tab("liquid", h), tab("liquid_norec", h)),
              (f"+-{h:g} s: random features - no liquid", tab("random_features", h), tab("no_liquid", h))]
for m in ("liquid", "random_features"):     # does the layer gain more from the long window than no layer does?
    items.append((f"({LAB4[m]} - no liquid) at +-{long_:g} s - the same at +-{short:g} s (does the layer use the extra timing?)",
                  diff_table(tab(m, long_), tab("no_liquid", long_)), diff_table(tab(m, short), tab("no_liquid", short))))
C4 = compare_family(f"{TASK}", items)
K4.to_csv(tpath("means") + ".csv", index=False); C4.to_csv(tpath("comparisons") + ".csv", index=False)
log_comparisons(C4)
_L = ["# Task 4: a longer window (the neighbouring beats) and the ladder", "",
      f"Config 11, the paper's liquid with the three fixes. Windows R +- {', '.join(f'{h:g}' for h in WINDOW_HALF_S)} s, "
      f"cut from the same filtered records; the same {int(OK1.sum())} DS1 and {int(OK2.sum())} DS2 beats for every size "
      f"(those whose longest window fits in the record), the authors' folds restricted to them. Seeds "
      f"{', '.join(map(str, TASK_SEEDS))}; DS2 and the {len(SPLITS_CV)} CV splits.", "",
      "* **no liquid**: the encoder's spikes counted in windows of 20 steps, straight to the readout.",
      "* **random features**: those counts times the liquid's input weights, centred, ReLU.",
      "* **liquid, no internal connections**: the spiking liquid with its recurrent connections switched off.",
      "* **liquid**: the full liquid.", "",
      f"Check against the replication's stored runs: **{CHECK['verdict'] if CHECK else 'not run'}**", "",
      "## Means", "", md_table(K4, MEANS_COLS), "",
      "## Paired comparisons (Macro-F1)", "",
      "Nadeau-Bengio corrected t-test over the CV splits, Holm over the family; DS2 difference averaged over seeds. "
      "The last rows compare how much each layer adds over no liquid at the long window with how much it adds at the short one.", "",
      md_table(C4, COMP_COLS)]
open(tpath("summary") + ".md", "w").write("\n".join(_L) + "\n")
log(f"{TASK}: summary at {tpath('summary')}.md")


## 11. End of session

In [ ]:
print("results:", TASK_DIR)
print("summary:", tpath("summary") + ".md")
print(f"session time: {(time.time() - SESSION_START) / 3600:.2f} h")
